# 1. MISALIGNMENT MANIPULATION CHECK — does real audio↔IMU misalignment hurt Condition D at all? — AeroSync-Mamba v26

**Question.** Does real cross-modal misalignment of up to ±1.0 s measurably degrade **Condition D** (no HTT time
embedding, no LACA lag bias) on TII? If D does not degrade, an alignment mechanism has nothing to recover on this
dataset. This is a small **screening check, not a sweep**.

**Design.**
- Dataset: **CORRECTED_ALIGNMENT** (95 missions, audio anchored at `recording_started_time`, same split), built by
  v24's unchanged data cells. OLD_ALIGNMENT is still built, because the CORRECTED cell depends on it, but it is never
  trained or evaluated.
- Models are trained on **normal, aligned data only**. Only the **evaluation input** is shifted. Nothing is trained at
  a nonzero offset.
- Conditions: **D** = HTT-time OFF, LACA-lag OFF. **C** = HTT-time ON, LACA-lag ON, with β = 1.0 fixed, no warm-up and
  v18's mlp + grid-LayerNorm lag bias, exactly as in v22/v25. Seeds 42, 123, 2024.

**Step 1: baselines at 0 ms.**
- **D:** v24's saved checkpoints (`v24_checkpoints/CORRECTED_ALIGNMENT_D_seed{s}_best.pt`) are reused if attached and
  if they reproduce v24's macro-F1 (0.9014 / 0.8846 / 0.8966, ±0.0005). Otherwise D is retrained. PASS/FAIL is printed
  per seed.
- **C:** trained for all 3 seeds. It is a **new baseline**, not a reproduction. It is shown next to v25's OLD-alignment C
  (0.8900 / 0.8838 / 0.8742) with a warning if it is more than 0.03 away.

**Step 2: shifted evaluation.**
- Offsets: 0, +0.5, −0.5, +1.0, −1.0 s.
- The sensor window is **re-sliced from the raw IMU log** at (audio window time + offset), with v25's `content_shift`
  code, v18's sampling to Ts = 128 and the CORRECTED training-set z-score. The audio is untouched.
- A window is dropped only if the shifted range leaves the raw IMU stream. Everything is scored on the **matched set**:
  windows valid at all 5 offsets.
- C is evaluated with **honest τ** (primary) and **re-zeroed τ** (diagnostic).

**Step 3.** Drops from each model's own 0 ms score, paired (D drop − C drop), and a **pre-registered** 3-branch verdict
at ±1.0 s.

**Sign convention.** **POSITIVE** = IMU delayed relative to audio: the sensor data comes from *later* than the audio
window. **NEGATIVE** = audio delayed relative to IMU: the sensor data comes from *earlier*.

Outputs: `manipulation_check_results.csv` (a row after every evaluation), `manipulation_check_training.csv`,
`manipulation_check_drops.csv`, `v26_checkpoints/`.

## 2. Configuration constants & environment
Every fixed value lives in the first cell below. The second installs `mamba_ssm` and reports which fusion backbone is active.

In [ ]:
import copy
import csv
import gc
import glob
import hashlib
import importlib.util
import inspect
import json
import math
import os
import random
import shutil
import subprocess
import sys
import textwrap
import time
import urllib.request
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# ==============================================================================
# FIXED CONFIGURATION — every finalized choice (see the summary table in v22)
# ==============================================================================
# --- v26: misalignment MANIPULATION CHECK on CORRECTED_ALIGNMENT (evaluation-time content shift only) ---
SEEDS = [42, 123, 2024]                  # 3 seeds: a screening check, not a final statistical claim
CONDITIONS = {                           # name: (HTT_TIME_EMBED_ENABLED, LACA_LAG_BIAS_ENABLED) — v22's switches
    "C": (True, True),                   # HTT time embedding ON, LACA lag bias ON (beta 1.0 fixed, no warm-up, v18 mlp+LN)
    "D": (False, False),                 # both OFF = v22/v24 Condition D
}
RUN_ORDER = [(seed, cond) for seed in SEEDS for cond in ("D", "C")]   # per seed: D (reuse or retrain), then C (train)
ALIGNMENT = "CORRECTED_ALIGNMENT"        # the ONLY dataset trained or evaluated here (OLD is built only as v24 does)
ALIGNMENTS = [ALIGNMENT]
SPLIT_SEED = 42                          # mission-level train/val/test split (independent of the training seed)

# --- v26: evaluation-time offsets (REAL seconds) and timestamp modes ---
# SIGN CONVENTION: POSITIVE offset = IMU delayed relative to audio -> the sensor window is re-sliced from the RAW IMU log
# at (audio window time + offset). NEGATIVE offset = audio delayed relative to IMU -> sensor data from EARLIER.
# The audio window, its content and tau_a never change. Only the sensor CONTENT (and its honest tau_s) moves.
OFFSETS_S = [0.0, +0.5, -0.5, +1.0, -1.0]
DECISION_OFFSETS_S = [+1.0, -1.0]        # the verdict is decided at +/-1.0 s; +/-0.5 s is informational
TAU_MODES = ["honest", "rezeroed"]       # honest = true time of the shifted samples vs the AUDIO window start (primary);
                                         # rezeroed = tau_s as a normal window starting at (t_start + offset) (C diagnostic)
RESLICE_TOLERANCE = 1e-6                 # offset-0 re-slice vs stored windows: max |difference| allowed (expected: 0 exactly)
N_SAMPLE_WINDOWS = 3                     # windows printed per nonzero offset (sensor range retrieved vs audio range)

# --- v26: pre-registered decision thresholds (fixed before any result is seen; do NOT tune) ---
DEGRADE_MIN_DROP = 0.015                 # D "degrades": mean macro-F1 drop >= this AND > SD of drops AND > 0 on all seeds
RETAIN_MIN_DIFF = 0.015                  # C "retains more": mean (D drop - C drop) >= this AND > SD AND > 0 on all seeds
C_SANITY_MAX_ABS = 0.03                  # C (new baseline) warned if > this away from v25's OLD-alignment C (context only)

# --- Dataset / split ---
DATASET_REPO_URL = "https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git"
WORK_DIR = Path("/kaggle/working")
DATASET_BASE = WORK_DIR / "tii_uav_dataset"
DATASET_ROOT = DATASET_BASE / "Dataset"
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
N_CLASSES = 5                            # propeller-fault severity 0..4

# --- Windowing ---
WINDOW_SEC = 2.56
STEP_SEC = 1.28
MIN_SENSOR_SAMPLES_PER_WINDOW = 10

# --- Audio: strongest-64 bins (train-only), sqrt(|rFFT|), no window, no normalization ---
N_AUDIO_BINS = 64                        # frequency axis
TA_AUDIO = 64                            # audio time positions per window

# --- Sensor: raw 6 channels (accel x/y/z, gyro x/y/z), no derived features ---
N_SENSOR_CHANNELS = 6
TS_SENSOR = 128                          # sensor time positions per window
SENSOR_STD_EPS = 1e-6

# --- Model ---
D_MODEL = 256
TIME_EMBED_FREQUENCIES = 64
LACA_NUM_HEADS = 4
LACA_BETA = 1.0                          # beta when LACA_LAG_BIAS_ENABLED (fixed from epoch 1: no warm-up, not learnable)
LAG_BIAS_HIDDEN = 32                     # mlp g(Δτ): Linear(1,32) -> Tanh -> Linear(32,1) = 97 params
LAG_BIAS_PARAMS = 97                     # the LayerNorm added in v18 has no parameters -> still 97
LAG_BIAS_LN_EPS = 1e-5                   # LayerNorm over each window's (T_m, T_n) lag-bias grid, no affine
MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
EXPECTED_MAMBA_PARAMS = 874_496          # 2 x 437,248 (real mamba_ssm)
DROPOUT = 0.15

# --- Loss: L_cls (class-weighted CE) + 0.1 * L_align (InfoNCE) + 0.1 * L_cons (KL) ---
LAMBDA_ALIGN = 0.1
LAMBDA_CONS = 0.1
INFONCE_TEMPERATURE = 0.07

# --- Optimization ---
EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LR_MIN = 1e-6
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2

# --- v23: corrected temporal anchor (CORRECTED_ALIGNMENT only — OLD_ALIGNMENT reads none of this) ---
# audio_start_offset_s = (recording_started_time - sensor_t0) / 1e6, both raw microsecond timestamps of the same clock
AUDIO_EXCLUDE_RATIO = 0.5                # exclude a mission if audio_start_offset_s / mission_duration_s >= this
EXPECTED_EXCLUDED_MISSIONS = {"rosbag2_2021_12_29-09_10_31", "rosbag2_2021_12_31-10_12_31",
                              "rosbag2_2021_12_31-09_58_13", "rosbag2_2021_12_31-10_04_22"}
EXPECTED_N_MISSIONS = {"OLD_ALIGNMENT": 99, "CORRECTED_ALIGNMENT": 95}
KNOWN_AUDIO_OFFSETS_S = {"rosbag2_2021_12_22-09_14_31": 24.38}   # earlier investigation (motor-start event at t = 24.5 s)
KNOWN_OFFSET_TOLERANCE_S = 0.05
EXPECTED_MEDIAN_OFFSET_S = 19.2          # median over all missions, earlier investigation
MEDIAN_OFFSET_TOLERANCE_S = 1.0
N_OFFSETS_TO_PRINT = 8
# Where CORRECTED windows start: "audio_onset" = the 2.56 s / 1.28 s grid starts at the mission's audio onset, so every
# window lies where BOTH modalities were recorded (v18 already ends the grid at min(sensor end, audio end)).
# "sensor_t0" = v18's grid start (t = 0); windows before the onset then get all-zero audio (np.interp left=0.0).
CORRECTED_WINDOW_START = "audio_onset"
# Strongest-64 bins for CORRECTED: "own_train" = recomputed (same method) from CORRECTED's own training missions;
# "old_bins" = reuse OLD_ALIGNMENT's frozen bins.
CORRECTED_BIN_SOURCE = "own_train"
assert CORRECTED_WINDOW_START in ("audio_onset", "sensor_t0") and CORRECTED_BIN_SOURCE in ("own_train", "old_bins")

# --- Outputs ---
CHECKPOINT_DIR = WORK_DIR / "v26_checkpoints"
PROBS_DIR = WORK_DIR / "v26_test_probs"
RESULTS_CSV = WORK_DIR / "manipulation_check_results.csv"      # one row per (condition, seed, offset, tau mode, eval set)
TRAINING_CSV = WORK_DIR / "manipulation_check_training.csv"    # Step 1: one row per model (reused or trained)
for _d in (CHECKPOINT_DIR, PROBS_DIR):
    _d.mkdir(parents=True, exist_ok=True)
# Where v24's CORRECTED_ALIGNMENT Condition D checkpoints may be (v24 saved them as
# v24_checkpoints/CORRECTED_ALIGNMENT_D_seed{seed}_best.pt): attached Kaggle inputs, or this session's working dir
# (including a previous v26 run in the same session).
D_CHECKPOINT_GLOBS = ["/kaggle/input/**/v24_checkpoints/{name}", "/kaggle/input/**/v26_checkpoints/{name}",
                      "/kaggle/input/**/{name}",
                      str(WORK_DIR / "v24_checkpoints" / "{name}"), str(WORK_DIR / "v26_checkpoints" / "{name}")]

# --- Reference numbers ---
SAME_SEED_TOLERANCE = 0.0005             # same-seed reproducibility (code correctness), not a finding threshold
SEED_TO_SEED_STD_F1 = 0.0160             # v15/v16 measured seed-to-seed std of test macro-F1
REFERENCE_SEED42_RNG_PREFIX = "5641b401720d"   # torch CPU RNG right after seed-42 model construction (v10-v22)
# Initial weights / torch RNG right after construction, per seed (v18 CSV). Identical for every v22 condition (both
# temporal modules are always built), so C and D of a seed start from the same weights.
SEED_INIT_REFERENCE = {42: ("5850841e699f", "5641b401720d"), 123: ("74be0883c62b", "0a4b3be44b18"),
                       2024: ("92852955eef1", "619c0e0a6b4d"), 7: ("7f5d0cd2e7cb", "9fdcf777d285"),
                       99: ("e40c78169f57", "81dcd4638a88")}
# v24 CORRECTED_ALIGNMENT, Condition D (Artifacts/Version 24/alignment_5seed_paired_results.csv, "own test set").
# THIS is what Step 1's D must reproduce: macro-F1 within ±SAME_SEED_TOLERANCE.
V24_CORRECTED_D = {
    42: {"accuracy": 0.9079754601226994, "macro_f1": 0.9014428634096197, "balanced_accuracy": 0.8939724876561621,
         "macro_auroc": 0.992763216249706, "class3_recall": 0.9553072625698324, "class4_recall": 0.7468354430379747,
         "confusion_matrix": [[212, 2, 0, 0, 0], [0, 153, 2, 1, 0], [0, 2, 86, 19, 1], [0, 0, 3, 171, 5], [0, 1, 0, 39, 118]],
         "best_epoch": 15, "train_seconds": 153.4},
    123: {"accuracy": 0.894478527607362, "macro_f1": 0.8846220280231434, "balanced_accuracy": 0.8763897171092857,
          "macro_auroc": 0.9881395753092281, "class3_recall": 0.88268156424581, "class4_recall": 0.8037974683544303,
          "confusion_matrix": [[213, 1, 0, 0, 0], [0, 154, 1, 0, 1], [0, 3, 77, 25, 3], [0, 0, 2, 158, 19], [0, 1, 1, 29, 127]],
          "best_epoch": 19, "train_seconds": 233.5},
    2024: {"accuracy": 0.9042944785276074, "macro_f1": 0.8965558896562792, "balanced_accuracy": 0.897410013106599,
           "macro_auroc": 0.9836323407488198, "class3_recall": 0.7821229050279329, "class4_recall": 0.8734177215189873,
           "confusion_matrix": [[213, 1, 0, 0, 0], [0, 155, 1, 0, 0], [0, 2, 91, 15, 0], [0, 0, 14, 140, 25], [0, 0, 2, 18, 138]],
           "best_epoch": 5, "train_seconds": 162.9},
}
# v25 / v22 Condition C on OLD_ALIGNMENT (99 missions, 1130 test windows). A DIFFERENT dataset: printed next to the new
# CORRECTED C only as a sanity range, NOT a reproduction target (no CORRECTED C has ever been trained before v26).
V25_OLD_ALIGNMENT_C_F1 = {42: 0.8900430576358016, 123: 0.8838406391126263, 2024: 0.8742357339300486}
REFERENCE_DATA = {
    "bin_indices": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255,
                    258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508,
                    509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760,
                    761, 762, 763, 764, 765, 766],
    "sensor_fingerprint": {
        "train": (4934, -6167105.225, 808427.456, (906, 1020, 1004, 1008, 996)),
        "val":   (954, -1193206.652, 156301.664, (126, 229, 199, 190, 210)),
        "test":  (1130, -1413563.944, 185127.438, (286, 205, 210, 228, 201)),
    },
}
# sha1[:16] of the source of every v18 data function / method (textwrap.dedent + rstrip). The two renamed functions are
# hashed under their v18 names. OLD_ALIGNMENT's code is checked against these: identical hash = identical code.
V18_DATA_SOURCE_SHA1 = {
    "parse_mission": "48132d5e84609fc4",
    "split_missions": "e60744ab660df986",
    "select_strongest_bins": "fb7d15f798514139",
    "audio_features": "d957dda051fb003d",
    "slice_windows": "0d493fd15803e0de",
    "UAVWindowDataset.__init__": "4a7708aec1baca55",
    "UAVWindowDataset.__len__": "15a9aa16ef54ea0a",
    "UAVWindowDataset.__getitem__": "5b6e2898c53b5989",
    "split_fingerprint": "195d93e4771675ca",
}
# CORRECTED_ALIGNMENT as v23 built it (v23 run output) + v23's CORRECTED function sources (sha1, same scheme as v18's).
V23_CORRECTED_DATA = {"n_missions": 95, "missions": {"train": 67, "val": 13, "test": 15},
                      "windows": {"train": 3737, "val": 686, "test": 815},
                      "class_counts": {"train": (701, 765, 706, 796, 769), "test": (214, 156, 108, 179, 158)}}
V23_CORRECTED_SOURCE_SHA1 = {"parse_mission_CORRECTED_ALIGNMENT": "49d4b2f5a1359025", "slice_windows_CORRECTED_ALIGNMENT": "8cf47da958eda742"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


def set_all_seeds(seed: int) -> None:
    """Seed python `random`, numpy, torch CPU and every CUDA device; deterministic cuDNN."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


print("█" * 110)
print("█  SIGN CONVENTION (every offset in this notebook, in REAL seconds)")
print("█    POSITIVE offset = IMU DELAYED relative to audio: the sensor window is re-sliced from the RAW IMU log at")
print("█                      (audio window time + offset), i.e. sensor data from LATER than the audio window.")
print("█    NEGATIVE offset = AUDIO DELAYED relative to IMU: the sensor window comes from EARLIER than the audio window.")
print("█    The audio window, its content and tau_a never change. The sensor CONTENT changes (not just its timestamps).")
print(f"█    Offsets tested: {OFFSETS_S} s | verdict decided at {DECISION_OFFSETS_S} s | +/-0.5 s informational")
print("█" * 110)
print(f"MANIPULATION CHECK: conditions {list(CONDITIONS)} x seeds {SEEDS} on {ALIGNMENT} | models trained on ALIGNED data "
      f"only; misalignment injected at EVALUATION only | tau modes {TAU_MODES} | results -> {RESULTS_CSV}")
print(f"Fixed configuration: audio strongest-{N_AUDIO_BINS}, Ta={TA_AUDIO}, sqrt, no window, no normalization "
      f"| sensor raw {N_SENSOR_CHANNELS} ch, Ts={TS_SENSOR} | LACA {LACA_NUM_HEADS} heads, beta {LACA_BETA} (C) / 0.0 (D) | "
      f"Mamba {MAMBA_NUM_BLOCKS} x (d_state {MAMBA_D_STATE}, d_conv {MAMBA_D_CONV}) | loss CE(class-weighted) + "
      f"{LAMBDA_ALIGN} InfoNCE + {LAMBDA_CONS} KL | {EPOCHS} epochs, batch {BATCH_SIZE}, AdamW {LEARNING_RATE}")
print(f"Pre-registered thresholds: D degrades if mean macro-F1 drop >= {DEGRADE_MIN_DROP} AND > SD AND > 0 on all "
      f"{len(SEEDS)} seeds | C retains more if mean (D drop - C drop) >= {RETAIN_MIN_DIFF} AND > SD AND > 0 on all seeds")

In [ ]:
# ==============================================================================
# mamba_ssm installation + fusion-backbone resolution.
# Order: an already-installed package, else a prebuilt wheel matched to this
# Python / torch / CUDA / C++ ABI (the reliable path on Kaggle), else a
# time-capped source build. If none succeeds, or there is no CUDA GPU, the
# pure-PyTorch fallback block is used and announced below.
# ==============================================================================
PIP_TIMEOUT_S = 240


def _env_tags():
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    try:
        cxx11_abi = bool(torch._C._GLIBCXX_USE_CXX11_ABI)
    except AttributeError:
        cxx11_abi = False
    return py_tag, torch_mm, torch.version.cuda, cxx11_abi


def _prebuilt_wheel_url(repo, py_tag, torch_mm, cuda_ver, cxx11_abi):
    """Best-matching .whl asset in `repo`'s GitHub releases (exact CUDA version first, then CUDA major)."""
    req = urllib.request.Request(f"https://api.github.com/repos/{repo}/releases",
                                 headers={"User-Agent": "kaggle-notebook", "Accept": "application/vnd.github+json"})
    try:
        with urllib.request.urlopen(req, timeout=20) as resp:
            releases = json.load(resp)
    except Exception as exc:
        print(f"    could not query github.com/{repo}/releases: {exc!r}")
        return None
    abi = f"cxx11abi{'TRUE' if cxx11_abi else 'FALSE'}"
    cuda_exact = f"cu{cuda_ver.replace('.', '')}"
    cuda_major = f"cu{cuda_ver.split('.')[0]}"
    exact, major_only = [], []
    for rel in releases:
        for asset in rel.get("assets", []):
            name = asset.get("name", "")
            if not (name.endswith(".whl") and py_tag in name and abi in name and f"torch{torch_mm}" in name):
                continue
            if cuda_exact in name:
                exact.append(asset["browser_download_url"])
            elif cuda_major in name:
                major_only.append(asset["browser_download_url"])
    chosen = exact or major_only
    return chosen[0] if chosen else None


def _pip_install(args):
    """pip with a hard timeout and captured output, so a stuck build can never hang the kernel."""
    try:
        proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args,
                              timeout=PIP_TIMEOUT_S, capture_output=True, text=True, check=False)
    except subprocess.TimeoutExpired:
        print(f"    pip install {args} exceeded {PIP_TIMEOUT_S}s — killed")
        return False
    if proc.returncode != 0:
        print(f"    pip install {args} failed (exit {proc.returncode}): {(proc.stderr or proc.stdout)[-400:]}")
    return proc.returncode == 0


def install_mamba_ssm():
    if importlib.util.find_spec("mamba_ssm") is not None:
        return "already installed"
    tags = _env_tags()
    print(f"Installing mamba_ssm + causal-conv1d (python={tags[0]}, torch={tags[1]}, cuda={tags[2]}, cxx11abi={tags[3]})")
    urls = [_prebuilt_wheel_url(repo, *tags) for repo in ("Dao-AILab/causal-conv1d", "state-spaces/mamba")]
    if all(urls) and all(_pip_install([u, "--no-deps"]) for u in urls):
        return "prebuilt wheel"
    print("  no complete prebuilt-wheel match — trying a time-capped source build")
    if _pip_install(["causal-conv1d>=1.2.0", "--no-build-isolation"]) and _pip_install(["mamba-ssm", "--no-build-isolation"]):
        return "source build"
    return "failed"


MAMBA_CLASS = None
if torch.cuda.is_available():
    print(f"mamba_ssm install: {install_mamba_ssm()}")
    try:
        from mamba_ssm import Mamba as MAMBA_CLASS
    except ImportError as exc:
        print(f"mamba_ssm import failed: {exc!r}")

if MAMBA_CLASS is not None:
    FUSION_BACKBONE = "mamba_ssm.Mamba (real selective-scan SSM, CUDA)"
    print(f"✅ FUSION BACKBONE: {FUSION_BACKBONE}")
else:
    FUSION_BACKBONE = "FALLBACK pure-PyTorch gated-conv block (NOT a state-space model)"
    print("🚨" * 45)
    print(f"🚨  FUSION BACKBONE: {FUSION_BACKBONE}")
    print("🚨  " + ("no CUDA GPU detected" if not torch.cuda.is_available() else "mamba_ssm could not be installed/imported")
          + " — results will NOT match the reference numbers, which were produced with real mamba_ssm.")
    print("🚨" * 45)

## 3a. Data — OLD_ALIGNMENT (v18's preprocessing, unchanged)
This is v18's data cell. Two functions are renamed and nothing else changes: `parse_mission` becomes
`parse_mission_OLD_ALIGNMENT`, and `slice_windows` becomes `slice_windows_OLD_ALIGNMENT`. All 99 missions are used, with
audio anchored at t = 0. At the end of the cell, every data function's source is hashed and compared with v18's, and
the bins and per-split fingerprints are compared with v18's reference values. Either a code change or a data change
would show up.

**v24 addition (bookkeeping only).** For each OLD window, the build loop now records its start time. It recomputes
the start with the same grid and skip rule that `slice_windows_OLD_ALIGNMENT` uses, and asserts that the recomputed
starts agree with every window's real timestamps. The time-matched control needs these starts. No function is changed,
and the v18 code and data checks below still apply.

**v26:** unchanged. OLD_ALIGNMENT is built only because the CORRECTED cell reuses its split, parser summary and bins. No
OLD model is trained or evaluated.

In [ ]:
# ==============================================================================
# DATA — OLD_ALIGNMENT: v18's preprocessing, UNCHANGED (audio anchored at t = 0, all 99 missions).
# parse missions -> mission-level split -> strongest-64 bins from TRAINING audio only (frozen) -> 2.56 s windows
# with real per-window timestamps -> datasets (sensor standardized with TRAINING stats).
# v23: parse_mission -> parse_mission_OLD_ALIGNMENT and slice_windows -> slice_windows_OLD_ALIGNMENT (renamed only).
# ==============================================================================
if not DATASET_ROOT.exists():
    print(f"Cloning {DATASET_REPO_URL} ...")
    subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO_URL, str(DATASET_BASE)], check=True)


def parse_mission_OLD_ALIGNMENT(mission_dir: Path):
    """Real 6-axis IMU (seconds from mission start) + raw audio chunks with per-chunk timestamps.
    Audio chunk duration = the mission's IMU-measured duration / number of chunks (both come from the same
    continuous recording; the audio log carries no per-chunk timestamp)."""
    sensor_files = list(mission_dir.glob("*SensorCombined*.jsonl"))
    audio_files = list(mission_dir.glob("*AudioBuffer*.jsonl"))
    if not sensor_files or not audio_files:
        return None

    sensor_times, accel, gyro = [], [], []
    with open(sensor_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                t_sec = float(rec["timestamp"]) / 1e6
                sensor_times.append(t_sec)
                accel.append(rec.get("accelerometer_m_s2", [0.0, 0.0, 9.81]))
                gyro.append(rec.get("gyro_rad", [0.0, 0.0, 0.0]))
            except Exception:
                continue
    if len(sensor_times) < 50:
        return None
    sensor_times = np.array(sensor_times, dtype=np.float64)
    sensor_times = sensor_times - sensor_times[0]
    telemetry = np.concatenate([np.array(accel, dtype=np.float32), np.array(gyro, dtype=np.float32)], axis=-1)

    audio_chunks = []
    with open(audio_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                chunk = rec.get("audio_data", rec.get("buffer", rec.get("data", [])))
                if len(chunk) > 0:
                    audio_chunks.append(chunk)
            except Exception:
                continue
    if len(audio_chunks) < 10:
        return None
    audio_chunks = np.array(audio_chunks, dtype=np.float32)

    chunk_duration_s = (sensor_times[-1] - sensor_times[0]) / len(audio_chunks)
    audio_times = np.arange(len(audio_chunks), dtype=np.float64) * chunk_duration_s
    return {"sensor_times": sensor_times, "telemetry": telemetry, "audio_times": audio_times, "audio_chunks": audio_chunks}


def split_missions(dataset_root: Path):
    """Mission-level stratified split (70 / 15 / 15 per class) with its own fixed seed."""
    class_missions = defaultdict(list)
    for class_dir in sorted(dataset_root.iterdir()):
        if not class_dir.is_dir() or class_dir.name.startswith("."):
            continue
        try:
            label = int(class_dir.name)
        except ValueError:
            continue
        for mission_dir in sorted(class_dir.iterdir()):
            if mission_dir.is_dir() and not mission_dir.name.startswith("."):
                class_missions[label].append(mission_dir)

    random.seed(SPLIT_SEED)
    splits = {"train": {}, "val": {}, "test": {}}
    for label in sorted(class_missions):
        missions = class_missions[label]
        random.shuffle(missions)
        n_train = max(1, int(len(missions) * TRAIN_FRACTION))
        n_val = max(1, int(len(missions) * VAL_FRACTION))
        splits["train"][label] = missions[:n_train]
        splits["val"][label] = missions[n_train:n_train + n_val]
        splits["test"][label] = missions[n_train + n_val:]
        print(f"Class {label} ({len(missions)} flights) -> train {n_train}, val {len(splits['val'][label])}, "
              f"test {len(splits['test'][label])}")
    return splits


def select_strongest_bins(train_missions, n_bins):
    """Indices of the n_bins rFFT magnitude bins with the highest mean magnitude over TRAINING audio only
    (unsupervised energy, no labels). Returned sorted; frozen and reused for val/test."""
    magnitude_sum, n_chunks, fft_len = None, 0, None
    for mission in train_missions:
        mag = np.abs(np.fft.rfft(mission["audio_chunks"], axis=1))
        fft_len = mag.shape[1] if fft_len is None else min(fft_len, mag.shape[1])
        mag = mag[:, :fft_len]
        if magnitude_sum is not None and len(magnitude_sum) > fft_len:
            magnitude_sum = magnitude_sum[:fft_len]
        magnitude_sum = mag.sum(axis=0) if magnitude_sum is None else magnitude_sum + mag.sum(axis=0)
        n_chunks += mag.shape[0]
    mean_magnitude = magnitude_sum / n_chunks
    return np.sort(np.argsort(mean_magnitude)[::-1][:n_bins]).astype(int)


def audio_features(audio_chunks, bin_indices):
    """raw chunk -> rFFT (no window) -> |.| -> frozen strongest bins -> sqrt (no normalization)."""
    mag = np.abs(np.fft.rfft(audio_chunks, axis=1))
    valid = bin_indices[bin_indices < mag.shape[1]]
    feats = np.zeros((mag.shape[0], len(bin_indices)), dtype=np.float32)
    feats[:, :len(valid)] = np.sqrt(mag[:, valid]).astype(np.float32)
    return feats


def slice_windows_OLD_ALIGNMENT(mission, label, bin_indices):
    """2.56 s windows (1.28 s step). Sensor: TS_SENSOR real IMU samples chosen evenly by index inside the window;
    audio: TA_AUDIO real chunk timestamps chosen the same way (a window with fewer than 2 audio chunks — the
    recording edge — falls back to an even grid). tau_a / tau_s = those REAL timestamps relative to window start."""
    s_times, telemetry = mission["sensor_times"], mission["telemetry"]
    a_times = mission["audio_times"]
    a_feats = audio_features(mission["audio_chunks"], bin_indices)
    max_time = min(s_times[-1], a_times[-1])

    windows, t_start = [], 0.0
    while t_start + WINDOW_SEC <= max_time:
        t_end = t_start + WINDOW_SEC
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < MIN_SENSOR_SAMPLES_PER_WINDOW:
            t_start += STEP_SEC
            continue

        s_idx = np.where(s_mask)[0]
        tau_s_abs = s_times[s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]]
        x_s = np.zeros((TS_SENSOR, N_SENSOR_CHANNELS), dtype=np.float32)
        for c in range(N_SENSOR_CHANNELS):
            x_s[:, c] = np.interp(tau_s_abs, s_times[s_mask], telemetry[s_mask, c])

        a_mask = (a_times >= t_start) & (a_times <= t_end)
        if a_mask.sum() >= 2:
            a_idx = np.where(a_mask)[0]
            tau_a_abs = a_times[a_idx[np.linspace(0, len(a_idx) - 1, TA_AUDIO).round().astype(int)]]
        else:
            tau_a_abs = np.linspace(t_start, t_end, TA_AUDIO)
        x_a = np.zeros((TA_AUDIO, len(bin_indices)), dtype=np.float32)
        for c in range(len(bin_indices)):
            x_a[:, c] = np.interp(tau_a_abs, a_times, a_feats[:, c], left=0.0, right=0.0)

        windows.append({
            "x_a": x_a,
            "x_s": x_s,
            "label": int(label),
            "tau_a": (tau_a_abs - t_start).astype(np.float32),
            "tau_s": (tau_s_abs - t_start).astype(np.float32),
            "window_sec": np.float32(WINDOW_SEC),
        })
        t_start += STEP_SEC
    return windows


class UAVWindowDataset(Dataset):
    """Windows -> tensors. Sensor channels are standardized with mean/std that must come from the TRAINING
    split (computed here only when none are passed, i.e. for the training set). tau is normalized by the
    window length."""

    def __init__(self, windows, sensor_mean=None, sensor_std=None):
        self.windows = windows
        if sensor_mean is None or sensor_std is None:
            all_s = np.concatenate([w["x_s"] for w in windows], axis=0)
            sensor_mean = all_s.mean(axis=0, keepdims=True)
            sensor_std = all_s.std(axis=0, keepdims=True) + SENSOR_STD_EPS
        self.sensor_mean, self.sensor_std = sensor_mean, sensor_std

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        window_sec = float(w["window_sec"])
        return {
            "x_a": torch.tensor(w["x_a"], dtype=torch.float32),
            "x_s": torch.tensor((w["x_s"] - self.sensor_mean) / self.sensor_std, dtype=torch.float32),
            "tau_a": torch.tensor(w["tau_a"], dtype=torch.float32) / window_sec,
            "tau_s": torch.tensor(w["tau_s"], dtype=torch.float32) / window_sec,
            "label": torch.tensor(w["label"], dtype=torch.long),
        }


def split_fingerprint(windows):
    """(n windows, sum of raw sensor values, sum of sensor timestamps, class counts) — deterministic data check."""
    x_s = np.stack([w["x_s"] for w in windows]).astype(np.float64)
    tau_s = np.stack([w["tau_s"] for w in windows]).astype(np.float64)
    counts = tuple(np.bincount([w["label"] for w in windows], minlength=N_CLASSES).tolist())
    return (len(windows), round(float(x_s.sum()), 3), round(float(tau_s.sum()), 3), counts)


# --- 1. split + parse ---
mission_splits = split_missions(DATASET_ROOT)
split_dirs = {s: [m for label in sorted(mission_splits[s]) for m in mission_splits[s][label]] for s in mission_splits}
assert not set(split_dirs["train"]) & (set(split_dirs["val"]) | set(split_dirs["test"])), "LEAKAGE: mission in two splits"
parsed_missions = {m: parse_mission_OLD_ALIGNMENT(m) for s in split_dirs for m in tqdm(split_dirs[s], desc=f"parse {s}", leave=False)}
# v23: per-mission raw-data summary, so the CORRECTED parser can be checked to read the same samples and chunks.
OLD_MISSION_SUMMARY = {m.name: (len(p["sensor_times"]), len(p["audio_chunks"]), float(p["sensor_times"][-1]))
                       for m, p in parsed_missions.items() if p is not None}

# --- 2. strongest-64 bins from TRAINING audio only ---
bin_source_missions = [m for m in split_dirs["train"] if parsed_missions[m] is not None]
assert set(bin_source_missions) <= set(split_dirs["train"])
SELECTED_BINS = select_strongest_bins([parsed_missions[m] for m in bin_source_missions], N_AUDIO_BINS)
assert len(SELECTED_BINS) == N_AUDIO_BINS
print(f"[LEAKAGE CHECK] strongest-{N_AUDIO_BINS} bins computed from {len(bin_source_missions)} TRAINING missions only "
      f"(0 of {len(split_dirs['val']) + len(split_dirs['test'])} val/test missions) -> frozen for all splits: "
      f"{SELECTED_BINS.tolist()}")

# --- 3. windows (the same frozen bins for every split) ---
def _old_window_starts(mission):
    """v24 bookkeeping: the t_start of every window slice_windows_OLD_ALIGNMENT returns, in order (same grid from 0.0,
    same += STEP_SEC accumulation, same end rule, same sensor-count skip). Does not build any window."""
    s_times, a_times = mission["sensor_times"], mission["audio_times"]
    max_time = min(s_times[-1], a_times[-1])
    starts, t_start = [], 0.0
    while t_start + WINDOW_SEC <= max_time:
        if (((s_times >= t_start) & (s_times <= t_start + WINDOW_SEC)).sum()) >= MIN_SENSOR_SAMPLES_PER_WINDOW:
            starts.append(t_start)
        t_start += STEP_SEC
    return starts


def _starts_match_windows(mission, windows, starts):
    """v24 bookkeeping check: rebuild each window's tau_s exactly as slice_windows_OLD_ALIGNMENT does (the samples inside
    [t, t + WINDOW_SEC] in FILE order, evenly chosen by index) from the recomputed start t, and compare with the stored
    tau_s. Valid whether or not the IMU timestamps are sorted."""
    s_times = mission["sensor_times"]
    for w, t in zip(windows, starts):
        s_idx = np.where((s_times >= t) & (s_times <= t + WINDOW_SEC))[0]
        tau_s = (s_times[s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]] - t).astype(np.float32)
        if not np.array_equal(tau_s, w["tau_s"]):
            return False
    return True


split_windows, split_window_missions = {}, {}   # v23: the mission of every window is recorded next to it (same order)
split_window_starts = {}                        # v24: ... and its start time (s from the first IMU sample)
NON_MONOTONIC_SENSOR_MISSIONS = {}              # v24 diagnostic: missions whose IMU timestamps go backwards somewhere
for split in ("train", "val", "test"):
    split_windows[split], split_window_missions[split], split_window_starts[split] = [], [], []
    for label in sorted(mission_splits[split]):
        for m in mission_splits[split][label]:
            if parsed_missions[m] is not None:
                _w = slice_windows_OLD_ALIGNMENT(parsed_missions[m], label, SELECTED_BINS)
                split_windows[split].extend(_w)
                split_window_missions[split].extend([m.name] * len(_w))
                _st = _old_window_starts(parsed_missions[m])
                assert len(_st) == len(_w), f"{m.name}: {len(_st)} recomputed starts vs {len(_w)} windows"
                assert _starts_match_windows(parsed_missions[m], _w, _st), \
                    f"{m.name}: recomputed window starts disagree with the windows' own timestamps"
                _back = int((np.diff(parsed_missions[m]["sensor_times"]) < 0).sum())
                if _back:
                    NON_MONOTONIC_SENSOR_MISSIONS[m.name] = _back
                split_window_starts[split].extend(_st)
    print(f"{split:<5}: {len(split_windows[split])} windows | audio {TA_AUDIO}x{N_AUDIO_BINS}, sensor {TS_SENSOR}x{N_SENSOR_CHANNELS}")
DATA_FINGERPRINT = {s: split_fingerprint(split_windows[s]) for s in split_windows}
print(f"[WINDOW STARTS] every OLD window's recomputed start verified by rebuilding its tau_s exactly ✅ | missions with "
      f"non-monotonic IMU timestamps: {len(NON_MONOTONIC_SENSOR_MISSIONS)}"
      + (f" {NON_MONOTONIC_SENSOR_MISSIONS} (backward steps per mission; v18's slicer picks samples in file order, so its "
         f"windows and these starts are unaffected — the rebuild above checks exactly that)" if NON_MONOTONIC_SENSOR_MISSIONS else ""))

# --- 4. datasets: sensor mean/std from the TRAINING split only, reused for val/test ---
train_ds = UAVWindowDataset(split_windows["train"])
val_ds = UAVWindowDataset(split_windows["val"], train_ds.sensor_mean, train_ds.sensor_std)
test_ds = UAVWindowDataset(split_windows["test"], train_ds.sensor_mean, train_ds.sensor_std)
_train_s = np.concatenate([w["x_s"] for w in split_windows["train"]], axis=0)
_val_s = np.concatenate([w["x_s"] for w in split_windows["val"]], axis=0)
assert val_ds.sensor_mean is train_ds.sensor_mean and test_ds.sensor_std is train_ds.sensor_std
assert np.array_equal(train_ds.sensor_mean, _train_s.mean(axis=0, keepdims=True))
print(f"[LEAKAGE CHECK] sensor standardization stats = TRAINING-split stats (same objects reused for val/test) | "
      f"train mean {np.round(train_ds.sensor_mean.ravel(), 3).tolist()} vs val's own mean "
      f"{np.round(_val_s.mean(axis=0), 3).tolist()} (not used)")
del _train_s, _val_s

# --- 5. real timestamps reach the model: tau must differ across windows ---
_tau_a = np.stack([train_ds[i]["tau_a"].numpy() for i in range(len(train_ds))])
_tau_s = np.stack([train_ds[i]["tau_s"].numpy() for i in range(len(train_ds))])
_n_unique_a, _n_unique_s = len(np.unique(_tau_a, axis=0)), len(np.unique(_tau_s, axis=0))
assert _n_unique_a > 1 and _n_unique_s > 1, "real timestamps are NOT reaching the model (tau identical for every window)"
assert not np.allclose(_tau_s, np.linspace(0.0, 1.0, TS_SENSOR)[None, :]), "tau_s equals a synthetic linspace grid"
print(f"[TIMESTAMP CHECK] {len(train_ds)} training windows: {_n_unique_a} distinct tau_a rows, {_n_unique_s} distinct "
      f"tau_s rows -> real per-window timestamps (Condition D does not use them)")
del _tau_a, _tau_s

# --- 6. v23: OLD_ALIGNMENT is v18, byte for byte (code) and value for value (data) ---
def _source_sha1(obj, v18_name=None):
    """sha1[:16] of obj's source (dedented, rstripped); a renamed function is compared under its v18 name."""
    s = textwrap.dedent(inspect.getsource(obj)).rstrip()
    if v18_name is not None:
        s = s.replace(f"def {obj.__name__}(", f"def {v18_name}(", 1)
    return hashlib.sha1(s.encode()).hexdigest()[:16]


_old_code = {"parse_mission": (parse_mission_OLD_ALIGNMENT, "parse_mission"),
             "split_missions": (split_missions, None),
             "select_strongest_bins": (select_strongest_bins, None),
             "audio_features": (audio_features, None),
             "slice_windows": (slice_windows_OLD_ALIGNMENT, "slice_windows"),
             "UAVWindowDataset.__init__": (UAVWindowDataset.__init__, None),
             "UAVWindowDataset.__len__": (UAVWindowDataset.__len__, None),
             "UAVWindowDataset.__getitem__": (UAVWindowDataset.__getitem__, None),
             "split_fingerprint": (split_fingerprint, None)}
assert set(_old_code) == set(V18_DATA_SOURCE_SHA1)
OLD_SOURCE_CHECK = {}
for _name, (_obj, _v18_name) in _old_code.items():
    try:
        _h = _source_sha1(_obj, _v18_name)
        OLD_SOURCE_CHECK[_name] = _h == V18_DATA_SOURCE_SHA1[_name]
        print(f"[OLD == v18 CODE] {_name:<30} sha1 {_h} vs v18 {V18_DATA_SOURCE_SHA1[_name]} -> "
              + ("identical ✅" if OLD_SOURCE_CHECK[_name] else "🚨 DIFFERENT — v18's preprocessing was modified"))
    except (OSError, TypeError) as _e:
        OLD_SOURCE_CHECK[_name] = None
        print(f"[OLD == v18 CODE] {_name:<30} source not readable here ({_e}); the data fingerprint below still applies")
OLD_DATA_CHECK = {"strongest-64 bin indices": SELECTED_BINS.tolist() == REFERENCE_DATA["bin_indices"],
                  **{f"{s} windows / sensor values / timestamps / class counts": DATA_FINGERPRINT[s] ==
                     REFERENCE_DATA["sensor_fingerprint"][s] for s in ("train", "val", "test")}}
for _name, _ok in OLD_DATA_CHECK.items():
    print(f"[OLD == v18 DATA] {_name:<58}: {'match ✅' if _ok else '🚨 MISMATCH'}")
OLD_N_MISSIONS = len(OLD_MISSION_SUMMARY)
print(f"[OLD MISSIONS] {OLD_N_MISSIONS} parsed missions (expected {EXPECTED_N_MISSIONS['OLD_ALIGNMENT']}) -> "
      + ("✅" if OLD_N_MISSIONS == EXPECTED_N_MISSIONS["OLD_ALIGNMENT"] else "🚨 UNEXPECTED COUNT"))
if not all(OLD_DATA_CHECK.values()) or False in OLD_SOURCE_CHECK.values():
    print("\n" + "🚨" * 40 + "\n🚨 OLD_ALIGNMENT does NOT reproduce v18's preprocessing — the OLD vs CORRECTED comparison "
          "is not against the historical pipeline.\n" + "🚨" * 40)
else:
    print("✅ OLD_ALIGNMENT = v18's preprocessing: identical code, identical strongest-64 bins, identical windows / sensor "
          "values / timestamps / class counts in every split. The 99-mission bin selection is v18's, unchanged.")

OLD_ALIGNMENT = {"name": "OLD_ALIGNMENT", "n_missions": OLD_N_MISSIONS, "bins": SELECTED_BINS,
                 "split_windows": split_windows, "window_missions": split_window_missions,
                 "window_starts": {s: np.array(v) for s, v in split_window_starts.items()},
                 "fingerprint": DATA_FINGERPRINT, "train_ds": train_ds, "val_ds": val_ds, "test_ds": test_ds,
                 "split_missions": {s: sorted({n for n in split_window_missions[s]}) for s in split_windows}}
del parsed_missions   # raw audio of 99 missions; the CORRECTED cell re-reads the raw files itself
gc.collect()

## 3b. Data — CORRECTED_ALIGNMENT (new, alongside OLD)
- **Parser.** `parse_mission_CORRECTED_ALIGNMENT` is a copy of the OLD parser with one change: the audio time axis is
  anchored at `recording_started_time`. That value is read from this mission's first AudioBuffer record, never
  hardcoded. `sensor_t0` is the raw timestamp of the first IMU sample, the same one OLD subtracts.
- **Checks.**
  - The first few offsets are printed, and `rosbag2_2021_12_22-09_14_31` must read ≈ 24.38 s.
  - The median offset must be ≈ 19.2 s.
  - `recording_started_time` must be constant within each mission.
  - The sample and chunk counts must equal OLD's.
- **Exclusion.** Missions with `offset / duration ≥ 0.5` are excluded from **every** split of this dataset only. They
  are printed with their ratios and checked against the 4 expected names.
- **Split, bins, windows.** The split is the same as OLD's. Bins are recomputed from CORRECTED's training missions, and
  windows start at the audio onset (see the header).
- **Summary.** Both datasets' sizes are printed side by side at the end.
- **v26 addition (bookkeeping only).** Before the parsed missions are freed, the raw IMU stream (`sensor_times`,
  `telemetry`) and `audio_times` of every CORRECTED **test** mission are kept in `CORRECTED_TEST_IMU`. The Step 2
  re-slicer reads from them. No function is changed, and the v23 hash checks below still apply.

In [ ]:
# ==============================================================================
# DATA — CORRECTED_ALIGNMENT: the SAME raw files and the SAME mission split, with the audio time axis anchored at the
# real recording start (recording_started_time) instead of t = 0. Missions whose audio starts in the second half of
# the flight are excluded from this dataset only. Nothing in the OLD_ALIGNMENT cell above is modified.
# ==============================================================================
def parse_mission_CORRECTED_ALIGNMENT(mission_dir: Path):
    """parse_mission_OLD_ALIGNMENT with ONE change — the audio time axis:
        OLD:       chunk_duration_s = mission_duration_s / n_chunks
                   audio_times      = arange(n_chunks) * chunk_duration_s
        CORRECTED: audio_start_offset_s = (recording_started_time - sensor_t0) / 1e6
                   chunk_duration_s     = (mission_duration_s - audio_start_offset_s) / n_chunks
                   audio_times          = audio_start_offset_s + arange(n_chunks) * chunk_duration_s
    recording_started_time: the FIRST AudioBuffer record's field (raw microseconds; constant within a mission).
    sensor_t0: raw microsecond timestamp of the first IMU sample (the sample OLD's sensor_times[0] refers to)."""
    sensor_files = list(mission_dir.glob("*SensorCombined*.jsonl"))
    audio_files = list(mission_dir.glob("*AudioBuffer*.jsonl"))
    if not sensor_files or not audio_files:
        return None

    sensor_times, accel, gyro, sensor_t0_us = [], [], [], None
    with open(sensor_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                t_sec = float(rec["timestamp"]) / 1e6
                sensor_times.append(t_sec)
                accel.append(rec.get("accelerometer_m_s2", [0.0, 0.0, 9.81]))
                gyro.append(rec.get("gyro_rad", [0.0, 0.0, 0.0]))
                if sensor_t0_us is None:
                    sensor_t0_us = float(rec["timestamp"])
            except Exception:
                continue
    if len(sensor_times) < 50:
        return None
    sensor_times = np.array(sensor_times, dtype=np.float64)
    sensor_times = sensor_times - sensor_times[0]
    telemetry = np.concatenate([np.array(accel, dtype=np.float32), np.array(gyro, dtype=np.float32)], axis=-1)

    audio_chunks, recording_started_time, rst_values = [], None, set()
    with open(audio_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                if "recording_started_time" in rec:
                    if recording_started_time is None:
                        recording_started_time = float(rec["recording_started_time"])
                    rst_values.add(float(rec["recording_started_time"]))
                chunk = rec.get("audio_data", rec.get("buffer", rec.get("data", [])))
                if len(chunk) > 0:
                    audio_chunks.append(chunk)
            except Exception:
                continue
    if len(audio_chunks) < 10:
        return None
    if recording_started_time is None:
        print(f"🚨 {mission_dir.name}: no recording_started_time in {audio_files[0].name} -> cannot anchor the audio; "
              f"mission left out of CORRECTED_ALIGNMENT")
        return None
    audio_chunks = np.array(audio_chunks, dtype=np.float32)

    mission_duration_s = float(sensor_times[-1] - sensor_times[0])
    audio_start_offset_s = (recording_started_time - sensor_t0_us) / 1e6
    chunk_duration_s = (mission_duration_s - audio_start_offset_s) / len(audio_chunks)
    audio_times = audio_start_offset_s + np.arange(len(audio_chunks), dtype=np.float64) * chunk_duration_s
    return {"sensor_times": sensor_times, "telemetry": telemetry, "audio_times": audio_times, "audio_chunks": audio_chunks,
            "recording_started_time": recording_started_time, "sensor_t0_us": sensor_t0_us,
            "n_distinct_recording_started_time": len(rst_values), "audio_start_offset_s": audio_start_offset_s,
            "mission_duration_s": mission_duration_s, "offset_ratio": audio_start_offset_s / mission_duration_s,
            "chunk_duration_s": chunk_duration_s, "old_chunk_duration_s": mission_duration_s / len(audio_chunks)}


def slice_windows_CORRECTED_ALIGNMENT(mission, label, bin_indices):
    """slice_windows_OLD_ALIGNMENT with ONE change — where the window grid starts:
    CORRECTED_WINDOW_START = "audio_onset": the first window starts at the audio onset (audio_times[0]), so every window
    lies inside the span where BOTH modalities were recorded (the end is min(sensor end, audio end), as in OLD).
    CORRECTED_WINDOW_START = "sensor_t0": OLD's start (t = 0); pre-onset windows then carry all-zero audio."""
    s_times, telemetry = mission["sensor_times"], mission["telemetry"]
    a_times = mission["audio_times"]
    a_feats = audio_features(mission["audio_chunks"], bin_indices)
    max_time = min(s_times[-1], a_times[-1])

    windows = []
    t_start = max(0.0, float(a_times[0])) if CORRECTED_WINDOW_START == "audio_onset" else 0.0
    while t_start + WINDOW_SEC <= max_time:
        t_end = t_start + WINDOW_SEC
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < MIN_SENSOR_SAMPLES_PER_WINDOW:
            t_start += STEP_SEC
            continue

        s_idx = np.where(s_mask)[0]
        tau_s_abs = s_times[s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]]
        x_s = np.zeros((TS_SENSOR, N_SENSOR_CHANNELS), dtype=np.float32)
        for c in range(N_SENSOR_CHANNELS):
            x_s[:, c] = np.interp(tau_s_abs, s_times[s_mask], telemetry[s_mask, c])

        a_mask = (a_times >= t_start) & (a_times <= t_end)
        if a_mask.sum() >= 2:
            a_idx = np.where(a_mask)[0]
            tau_a_abs = a_times[a_idx[np.linspace(0, len(a_idx) - 1, TA_AUDIO).round().astype(int)]]
        else:
            tau_a_abs = np.linspace(t_start, t_end, TA_AUDIO)
        x_a = np.zeros((TA_AUDIO, len(bin_indices)), dtype=np.float32)
        for c in range(len(bin_indices)):
            x_a[:, c] = np.interp(tau_a_abs, a_times, a_feats[:, c], left=0.0, right=0.0)

        windows.append({
            "x_a": x_a,
            "x_s": x_s,
            "label": int(label),
            "tau_a": (tau_a_abs - t_start).astype(np.float32),
            "tau_s": (tau_s_abs - t_start).astype(np.float32),
            "window_sec": np.float32(WINDOW_SEC),
        })
        t_start += STEP_SEC
    return windows


# --- 1. parse every mission of the SAME split (mission_splits / split_dirs from the OLD cell: SPLIT_SEED 42) ---
corrected_parsed = {m: parse_mission_CORRECTED_ALIGNMENT(m) for s in split_dirs
                    for m in tqdm(split_dirs[s], desc=f"parse {s} (corrected)", leave=False)}
mission_home = {m: (s, label) for s in mission_splits for label in mission_splits[s] for m in mission_splits[s][label]}
_ok = {m: p for m, p in corrected_parsed.items() if p is not None}
_same_raw = [m.name for m, p in _ok.items()
             if OLD_MISSION_SUMMARY.get(m.name) != (len(p["sensor_times"]), len(p["audio_chunks"]), float(p["sensor_times"][-1]))]
print(f"[RAW DATA CHECK] {len(_ok)} missions parsed by the CORRECTED parser; IMU samples, audio chunks and duration equal "
      f"OLD's for " + ("every mission ✅" if not _same_raw and len(_ok) == len(OLD_MISSION_SUMMARY) else
                       f"all but {_same_raw} 🚨"))
_non_const = {m.name: p["n_distinct_recording_started_time"] for m, p in _ok.items() if p["n_distinct_recording_started_time"] != 1}
RST_CONSTANT = not _non_const
print("[METADATA] recording_started_time is constant within every mission ✅" if RST_CONSTANT else
      f"🚨 [METADATA] recording_started_time varies within {_non_const} (distinct values per mission) — the first "
      f"record's value is used, as specified")

# --- 2. verification: offsets read from the metadata ---
_by_name = {m.name: (m, p) for m, p in _ok.items()}
print(f"\n[OFFSETS] first {N_OFFSETS_TO_PRINT} missions (by name): audio_start_offset_s = (recording_started_time - "
      f"sensor_t0) / 1e6")
print(f"    {'mission':<30} {'class':>5} {'split':>5} {'sensor_t0 (us)':>18} {'rec_started (us)':>18} {'offset s':>9} "
      f"{'dur s':>8} {'ratio':>6} {'chunk ms OLD':>12} {'chunk ms NEW':>12}")
for _name in sorted(_by_name)[:N_OFFSETS_TO_PRINT] + [n for n in KNOWN_AUDIO_OFFSETS_S if n in _by_name]:
    _m, _p = _by_name[_name]
    _s, _lab = mission_home[_m]
    print(f"    {_name:<30} {_lab:>5} {_s:>5} {_p['sensor_t0_us']:>18.0f} {_p['recording_started_time']:>18.0f} "
          f"{_p['audio_start_offset_s']:>9.3f} {_p['mission_duration_s']:>8.2f} {_p['offset_ratio']:>6.3f} "
          f"{1e3 * _p['old_chunk_duration_s']:>12.3f} {1e3 * _p['chunk_duration_s']:>12.3f}")
KNOWN_OFFSET_CHECK = {}
for _name, _expected in KNOWN_AUDIO_OFFSETS_S.items():
    if _name not in _by_name:
        KNOWN_OFFSET_CHECK[_name] = False
        print(f"🚨 [KNOWN OFFSET] {_name} not found among the parsed missions")
        continue
    _got = _by_name[_name][1]["audio_start_offset_s"]
    KNOWN_OFFSET_CHECK[_name] = abs(_got - _expected) <= KNOWN_OFFSET_TOLERANCE_S
    print(f"[KNOWN OFFSET] {_name}: computed {_got:.3f} s vs earlier investigation ~{_expected:.2f} s -> "
          + ("✅ metadata read correctly" if KNOWN_OFFSET_CHECK[_name] else f"🚨 MISMATCH (> ±{KNOWN_OFFSET_TOLERANCE_S} s)"))
_offsets = np.array([p["audio_start_offset_s"] for p in _ok.values()])
MEDIAN_OFFSET_S = float(np.median(_offsets))
MEDIAN_OFFSET_OK = abs(MEDIAN_OFFSET_S - EXPECTED_MEDIAN_OFFSET_S) <= MEDIAN_OFFSET_TOLERANCE_S
print(f"[OFFSETS] over {len(_offsets)} missions: median {MEDIAN_OFFSET_S:.2f} s (expected ~{EXPECTED_MEDIAN_OFFSET_S}) "
      f"{'✅' if MEDIAN_OFFSET_OK else '🚨'} | min {_offsets.min():.2f} s | max {_offsets.max():.2f} s | "
      f"{int((_offsets < 0).sum())} negative")
print(f"[OLD MISPLACEMENT] under OLD_ALIGNMENT, chunk k sits at k*D/n but was recorded at offset + k*(D - offset)/n: the "
      f"audio is placed offset*(1 - k/n) too EARLY — {MEDIAN_OFFSET_S:.1f} s at the start of a median mission, shrinking "
      f"to 0 at the end (mean {MEDIAN_OFFSET_S / 2:.1f} s = {MEDIAN_OFFSET_S / 2 / WINDOW_SEC:.1f} windows)")

# --- 3. exclusion rule: offset / duration >= AUDIO_EXCLUDE_RATIO -> out of CORRECTED_ALIGNMENT (every split) ---
_ratios = {n: p["offset_ratio"] for n, (m, p) in _by_name.items()}
EXCLUDED_MISSIONS = {n: r for n, r in sorted(_ratios.items()) if r >= AUDIO_EXCLUDE_RATIO}
print(f"\n[EXCLUSION] rule: audio_start_offset_s / mission_duration_s >= {AUDIO_EXCLUDE_RATIO} -> excluded from "
      f"CORRECTED_ALIGNMENT only ({len(EXCLUDED_MISSIONS)} missions):")
for _name, _r in EXCLUDED_MISSIONS.items():
    _m, _p = _by_name[_name]
    _s, _lab = mission_home[_m]
    print(f"    EXCLUDED {_name:<30} ratio {_r:.3f} | offset {_p['audio_start_offset_s']:.2f} s of "
          f"{_p['mission_duration_s']:.2f} s | class {_lab} | split {_s}")
print("    nearest kept missions (highest ratios below the threshold): " + ", ".join(
    f"{n} {r:.3f}" for n, r in sorted(((n, r) for n, r in _ratios.items() if r < AUDIO_EXCLUDE_RATIO), key=lambda t: -t[1])[:3]))
EXCLUSION_MATCHES = set(EXCLUDED_MISSIONS) == EXPECTED_EXCLUDED_MISSIONS
if EXCLUSION_MATCHES:
    print(f"[EXCLUSION] ✅ computed exclusion set == the 4 expected missions {sorted(EXPECTED_EXCLUDED_MISSIONS)}")
else:
    print("\n" + "🚨" * 40)
    print(f"🚨 EXCLUSION SET DOES NOT MATCH THE EXPECTED LIST\n🚨   computed but not expected: "
          f"{sorted(set(EXCLUDED_MISSIONS) - EXPECTED_EXCLUDED_MISSIONS)}\n🚨   expected but not computed: "
          f"{sorted(EXPECTED_EXCLUDED_MISSIONS - set(EXCLUDED_MISSIONS))} (ratios: "
          f"{ {n: round(_ratios[n], 3) for n in EXPECTED_EXCLUDED_MISSIONS if n in _ratios} })\n🚨 The computed set is "
          f"what is applied. Check the metadata before trusting this run.")
    print("🚨" * 40 + "\n")
corrected_splits = {s: {label: [m for m in mission_splits[s][label]
                                if corrected_parsed[m] is not None and m.name not in EXCLUDED_MISSIONS]
                        for label in mission_splits[s]} for s in mission_splits}
CORRECTED_N_MISSIONS = sum(len(v) for s in corrected_splits for v in corrected_splits[s].values())
print(f"[CORRECTED MISSIONS] {CORRECTED_N_MISSIONS} missions (expected {EXPECTED_N_MISSIONS['CORRECTED_ALIGNMENT']}) -> "
      + ("✅" if CORRECTED_N_MISSIONS == EXPECTED_N_MISSIONS["CORRECTED_ALIGNMENT"] else "🚨 UNEXPECTED COUNT"))

# --- 4. strongest-64 bins: from CORRECTED's TRAINING missions only (or OLD's frozen bins) ---
_corr_train = [m for label in sorted(corrected_splits["train"]) for m in corrected_splits["train"][label]]
assert set(_corr_train) <= set(split_dirs["train"])
if CORRECTED_BIN_SOURCE == "own_train":
    CORRECTED_BINS = select_strongest_bins([corrected_parsed[m] for m in _corr_train], N_AUDIO_BINS)
else:
    CORRECTED_BINS = OLD_ALIGNMENT["bins"].copy()
_overlap = len(set(CORRECTED_BINS.tolist()) & set(OLD_ALIGNMENT["bins"].tolist()))
CORRECTED_BINS_SAME = CORRECTED_BINS.tolist() == OLD_ALIGNMENT["bins"].tolist()
print(f"[BINS] CORRECTED strongest-{N_AUDIO_BINS} ({CORRECTED_BIN_SOURCE}: {len(_corr_train)} TRAINING missions, "
      f"0 val/test) -> {_overlap}/{N_AUDIO_BINS} shared with OLD -> "
      + ("identical to OLD's bins ✅" if CORRECTED_BINS_SAME else
         f"DIFFERENT: only in CORRECTED {sorted(set(CORRECTED_BINS.tolist()) - set(OLD_ALIGNMENT['bins'].tolist()))}, "
         f"only in OLD {sorted(set(OLD_ALIGNMENT['bins'].tolist()) - set(CORRECTED_BINS.tolist()))}"))

# --- 5. windows ---
corr_windows, corr_window_missions = {}, {}
for split in ("train", "val", "test"):
    corr_windows[split], corr_window_missions[split] = [], []
    for label in sorted(corrected_splits[split]):
        for m in corrected_splits[split][label]:
            _w = slice_windows_CORRECTED_ALIGNMENT(corrected_parsed[m], label, CORRECTED_BINS)
            corr_windows[split].extend(_w)
            corr_window_missions[split].extend([m.name] * len(_w))
    print(f"{split:<5}: {len(corr_windows[split])} windows | audio {TA_AUDIO}x{N_AUDIO_BINS}, sensor {TS_SENSOR}x{N_SENSOR_CHANNELS}")
CORRECTED_FINGERPRINT = {s: split_fingerprint(corr_windows[s]) for s in corr_windows}

# --- 6. datasets: sensor mean/std from CORRECTED's TRAINING split only ---
corr_train_ds = UAVWindowDataset(corr_windows["train"])
corr_val_ds = UAVWindowDataset(corr_windows["val"], corr_train_ds.sensor_mean, corr_train_ds.sensor_std)
corr_test_ds = UAVWindowDataset(corr_windows["test"], corr_train_ds.sensor_mean, corr_train_ds.sensor_std)
assert corr_val_ds.sensor_mean is corr_train_ds.sensor_mean and corr_test_ds.sensor_std is corr_train_ds.sensor_std
print(f"[LEAKAGE CHECK] CORRECTED sensor standardization = CORRECTED training-split stats | mean "
      f"{np.round(corr_train_ds.sensor_mean.ravel(), 3).tolist()}")

# --- 7. audio coverage: how many windows carry all-zero audio (no chunk inside them) ---
def _zero_audio(windows):
    rows = np.stack([np.abs(w["x_a"]).sum(axis=1) == 0 for w in windows])   # (n windows, TA_AUDIO)
    return int(rows.all(axis=1).sum()), int(rows.any(axis=1).sum())
ZERO_AUDIO = {}
for _tag, _wins in (("OLD_ALIGNMENT", OLD_ALIGNMENT["split_windows"]), ("CORRECTED_ALIGNMENT", corr_windows)):
    _all = [w for s in ("train", "val", "test") for w in _wins[s]]
    ZERO_AUDIO[_tag] = _zero_audio(_all)
    print(f"[AUDIO COVERAGE] {_tag:<20}: {ZERO_AUDIO[_tag][0]} of {len(_all)} windows have ALL-zero audio, "
          f"{ZERO_AUDIO[_tag][1]} have some all-zero audio rows")

CORRECTED_ALIGNMENT = {"name": "CORRECTED_ALIGNMENT", "n_missions": CORRECTED_N_MISSIONS, "bins": CORRECTED_BINS,
                       "split_windows": corr_windows, "window_missions": corr_window_missions,
                       "fingerprint": CORRECTED_FINGERPRINT, "train_ds": corr_train_ds, "val_ds": corr_val_ds,
                       "test_ds": corr_test_ds,
                       "split_missions": {s: sorted({n for n in corr_window_missions[s]}) for s in corr_windows},
                       "excluded": EXCLUDED_MISSIONS,
                       "offsets_s": {n: p["audio_start_offset_s"] for n, (m, p) in _by_name.items()}}
# --- v26: keep the RAW IMU stream (+ audio times) of every CORRECTED TEST mission for the Step 2 re-slicer ---
CORRECTED_TEST_IMU = {m.name: {"sensor_times": corrected_parsed[m]["sensor_times"],
                               "telemetry": corrected_parsed[m]["telemetry"],
                               "audio_times": corrected_parsed[m]["audio_times"]}
                      for label in sorted(corrected_splits["test"]) for m in corrected_splits["test"][label]}
print(f"[v26] raw IMU kept for {len(CORRECTED_TEST_IMU)} CORRECTED test missions (evaluation-time re-slicing)")
del corrected_parsed, _ok, _by_name
gc.collect()

# --- 8. both datasets, side by side ---
DATASETS = {"OLD_ALIGNMENT": OLD_ALIGNMENT, "CORRECTED_ALIGNMENT": CORRECTED_ALIGNMENT}
print("\n" + "=" * 100 + "\nDATASET SIZES — both built, both kept\n" + "=" * 100)
print(f"    {'':<22}" + "".join(f"{n:>24}" for n in DATASETS))
print(f"    {'missions (total)':<22}" + "".join(f"{d['n_missions']:>24}" for d in DATASETS.values()))
for _s in ("train", "val", "test"):
    print(f"    {'missions ' + _s:<22}" + "".join(f"{len(d['split_missions'][_s]):>24}" for d in DATASETS.values()))
for _s in ("train", "val", "test"):
    print(f"    {'windows ' + _s:<22}" + "".join(f"{len(d['split_windows'][_s]):>24}" for d in DATASETS.values()))
print(f"    {'windows (total)':<22}" + "".join(f"{sum(len(v) for v in d['split_windows'].values()):>24}" for d in DATASETS.values()))
for _s in ("train", "test"):
    print(f"    {'class counts ' + _s:<22}" + "".join(f"{str(d['fingerprint'][_s][3]):>24}" for d in DATASETS.values()))
for _name, _r in EXCLUDED_MISSIONS.items():
    print(f"    excluded from CORRECTED: {_name} (split {mission_home[[m for m in mission_home if m.name == _name][0]][0]}, "
          f"ratio {_r:.3f})")

## 3c. v23 reuse checks
The two CORRECTED functions are hashed and compared with v23's source. Mission, window and class counts are compared
with v23's run. v24's class-level diagnostic and OLD-vs-CORRECTED controls are dropped, since v26 does not compare
alignments.

In [ ]:
# ==============================================================================
# v23 REUSE CHECKS (CORRECTED code + data == v23) (v26: v24's diagnostic and controls removed)
# ==============================================================================
CORRECTED_SOURCE_CHECK = {}
for _name, _obj in (("parse_mission_CORRECTED_ALIGNMENT", parse_mission_CORRECTED_ALIGNMENT),
                    ("slice_windows_CORRECTED_ALIGNMENT", slice_windows_CORRECTED_ALIGNMENT)):
    try:
        _h = _source_sha1(_obj)
        CORRECTED_SOURCE_CHECK[_name] = _h == V23_CORRECTED_SOURCE_SHA1[_name]
        print(f"[CORRECTED == v23 CODE] {_name:<36} sha1 {_h} vs v23 {V23_CORRECTED_SOURCE_SHA1[_name]} -> "
              + ("identical ✅" if CORRECTED_SOURCE_CHECK[_name] else "🚨 DIFFERENT"))
    except (OSError, TypeError) as _e:
        CORRECTED_SOURCE_CHECK[_name] = None
        print(f"[CORRECTED == v23 CODE] {_name:<36} source not readable here ({_e}); the counts below still apply")
CORRECTED_DATA_CHECK = {
    "missions": CORRECTED_ALIGNMENT["n_missions"] == V23_CORRECTED_DATA["n_missions"],
    **{f"{s} missions": len(CORRECTED_ALIGNMENT["split_missions"][s]) == V23_CORRECTED_DATA["missions"][s] for s in ("train", "val", "test")},
    **{f"{s} windows": len(CORRECTED_ALIGNMENT["split_windows"][s]) == V23_CORRECTED_DATA["windows"][s] for s in ("train", "val", "test")},
    **{f"{s} class counts": tuple(CORRECTED_ALIGNMENT["fingerprint"][s][3]) == V23_CORRECTED_DATA["class_counts"][s] for s in ("train", "test")},
    "exclusion set == the 4 expected": EXCLUSION_MATCHES,
    "bins identical to OLD (as in v23)": CORRECTED_BINS_SAME,
}
for _name, _ok in CORRECTED_DATA_CHECK.items():
    print(f"[CORRECTED == v23 DATA] {_name:<36}: {'match ✅' if _ok else '🚨 MISMATCH'}")
DATA_REUSE_OK = (all(OLD_DATA_CHECK.values()) and False not in OLD_SOURCE_CHECK.values()
                 and all(CORRECTED_DATA_CHECK.values()) and False not in CORRECTED_SOURCE_CHECK.values())
print(("✅ Both preprocessing paths are reused exactly as built in v23 (OLD == v18, CORRECTED == v23): same code, same "
       "missions, same windows.") if DATA_REUSE_OK else
      "🚨 A preprocessing path differs from v23 — seed 42 will NOT be reused, and results are not comparable to v23.")

## 4. Model definition — v22 switches, Conditions C and D
This is the v22/v24 model cell, unchanged apart from the probe at the end, which now builds both conditions.
- **D** = `AeroSyncMamba(htt_time=False, laca_lag=False)`: HTT returns `norm(z + e_m)` and LACA adds `0.0 · LN_grid(g(Δτ))`.
- **C** = `AeroSyncMamba(htt_time=True, laca_lag=True)`: HTT adds `time_embedding(τ)`, and LACA adds
  `1.0 · LN_grid(g(Δτ))` from epoch 1, where g is v18's zero-initialized mlp.

Both modules are always built, so C and D of a seed start from the same weights and RNG stream (`SEED_INIT_REFERENCE`).

In [ ]:
# ==============================================================================
# MODEL — encoders -> HTT (modality + continuous-time embedding of the REAL tau)
# -> bi-directional LACA (lag-aware cross-attention; lag bias beta * LN_grid(g(Δτ)), beta = 1.0 when
#    LACA_LAG_BIAS_ENABLED else 0.0; 4 heads). v22: HTT's time embedding is switchable too.
# v26: the classes below are v22's, unchanged; both Condition C and Condition D are built.
# -> [H_a ; SEP ; H_s] -> 2 Mamba blocks -> attention pooling -> classifier.
# Auxiliary per-modality heads feed only the consistency loss.
# ==============================================================================
class ContinuousTimeEmbedding(nn.Module):
    def __init__(self, d_model: int, num_frequencies: int = TIME_EMBED_FREQUENCIES):
        super().__init__()
        self.frequencies = nn.Parameter(
            torch.exp(torch.linspace(0, math.log(1000.0), num_frequencies)), requires_grad=False
        )
        self.mlp = nn.Sequential(
            nn.Linear(2 * num_frequencies, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, tau: torch.Tensor) -> torch.Tensor:
        if tau.dim() == 2:
            tau = tau.unsqueeze(-1)
        angles = tau * self.frequencies * 2.0 * math.pi
        return self.mlp(torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1))


class HeterogeneousTemporalTokenization(nn.Module):
    """token = LayerNorm(features + modality embedding [+ time embedding(tau)]); modality 0 = audio, 1 = sensor.
    v22: time_enabled = HTT_TIME_EMBED_ENABLED. When False the time embedding is skipped entirely (no temporal
    position information enters through HTT); the module is still built so every condition draws the same RNG."""

    def __init__(self, d_model: int, time_enabled: bool = True):
        super().__init__()
        self.d_model = d_model
        self.time_enabled = time_enabled
        self.modality_embeddings = nn.Embedding(2, d_model)
        self.time_embedding = ContinuousTimeEmbedding(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, z: torch.Tensor, tau: torch.Tensor, modality_idx: int) -> torch.Tensor:
        mod_idx = torch.tensor(modality_idx, device=z.device, dtype=torch.long)
        e_m = self.modality_embeddings(mod_idx).view(1, 1, self.d_model)
        if not self.time_enabled:
            return self.norm(z + e_m)                       # HTT_TIME_EMBED_ENABLED = False
        return self.norm(z + e_m + self.time_embedding(tau))


class GridLayerNorm(nn.Module):
    """LayerNorm for the lag-bias grid (v18).

    g(Δτ) produces ONE scalar per (query i, key j) pair, so its output is a (B, T_m, T_n) grid with no feature
    dimension > 1. A LayerNorm over a size-1 dimension would subtract each value from itself and return 0 for every
    input. Instead, the normalization is taken over each window's whole (T_m, T_n) grid, i.e. every bias value that
    enters that window's attention matrix. It is computed separately for each window: every window has its own real
    timestamps, and mixing windows would make one window's bias depend on the rest of the batch and differ between
    training and evaluation. There are no affine parameters, so beta is the only source of scale.

    Consequences: the subtracted mean is one constant per window, and softmax is unchanged by a constant added to
    every score, so only the division by the std affects attention. For an all-zero grid (the zero-init state) the
    output is exactly 0. As g develops variance v, the output std grows smoothly as sqrt(v / (v + eps)) toward 1."""

    def __init__(self, eps: float = LAG_BIAS_LN_EPS):
        super().__init__()
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:   # x: (B, T_m, T_n)
        return F.layer_norm(x, x.shape[-2:], eps=self.eps)


class PairwiseLACA(nn.Module):
    """Cross-attention from modality m to n with a learned lag bias g(Δτ) added to the scores (scaled by β)."""

    def __init__(self, d_model: int, num_heads: int = LACA_NUM_HEADS, beta: float = LACA_BETA):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.beta = beta
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.lag_bias = nn.Sequential(nn.Linear(1, LAG_BIAS_HIDDEN), nn.Tanh(), nn.Linear(LAG_BIAS_HIDDEN, 1))
        # v18 ZERO-INIT: g's final Linear starts at exactly 0 -> g(Δτ) = 0 for every Δτ and every seed. The layer was
        # already built (same RNG draws as v16), and nn.init.zeros_ draws no random numbers, so every other
        # layer's initial weights and the training RNG stream are unchanged.
        nn.init.zeros_(self.lag_bias[-1].weight)
        nn.init.zeros_(self.lag_bias[-1].bias)
        self.lag_bias_norm = GridLayerNorm()   # v18: no parameters

    def lag_bias_term(self, tau_m, tau_n):
        """LN_grid(g(Δτ)), BEFORE beta. Returns (B, 1, T_m, T_n), broadcast over the heads."""
        delta_tau = (tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)   # (B, T_m, T_n, 1)
        g = self.lag_bias(delta_tau).squeeze(-1)                                # (B, T_m, T_n): one scalar per (i, j)
        return self.lag_bias_norm(g).unsqueeze(1)                               # normalized per window -> (B, 1, T_m, T_n)

    def forward(self, H_m, H_n, tau_m, tau_n):
        B, T_m, _ = H_m.shape
        _, T_n, _ = H_n.shape
        Q = self.w_q(H_m).view(B, T_m, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.w_k(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        lag_bias = self.lag_bias_term(tau_m, tau_n)

        attn = F.softmax(scores + self.beta * lag_bias, dim=-1)   # beta = 1.0 (lag bias ON) or exactly 0.0 (OFF)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, T_m, self.d_model)

        attn_mean = attn.mean(dim=1)
        tau_diff = tau_n.unsqueeze(1) - tau_m.unsqueeze(2)
        expected_lag = (attn_mean * tau_diff).sum(dim=(1, 2)) / float(T_m)
        return self.out_proj(out), expected_lag


class BiModalLACAFusion(nn.Module):
    def __init__(self, d_model: int, beta: float = LACA_BETA):
        super().__init__()
        self.laca_as = PairwiseLACA(d_model, beta=beta)   # beta scales the lag bias in BOTH directions
        self.laca_sa = PairwiseLACA(d_model, beta=beta)
        self.gamma_as = nn.Parameter(torch.tensor(0.5))
        self.gamma_sa = nn.Parameter(torch.tensor(0.5))

    def forward(self, Ha, Hs, tau_a, tau_s):
        H_a_from_s, lag_as = self.laca_as(Ha, Hs, tau_a, tau_s)
        H_s_from_a, lag_sa = self.laca_sa(Hs, Ha, tau_s, tau_a)
        return Ha + self.gamma_as * H_a_from_s, Hs + self.gamma_sa * H_s_from_a, {"lag_as": lag_as, "lag_sa": lag_sa}


class MambaBlock(nn.Module):
    """Pre-norm residual wrapper around the real mamba_ssm.Mamba selective-scan SSM."""

    def __init__(self, d_model: int, d_state: int, d_conv: int, expand: int):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = MAMBA_CLASS(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.mamba(self.norm(x))


class FallbackGatedConvBlock(nn.Module):
    """CPU-only stand-in when mamba_ssm is unavailable: depthwise causal conv + gated MLP. NOT a state-space
    model (no hidden state, no A/B/C, no discretization)."""

    def __init__(self, d_model: int, d_conv: int, expand: int):
        super().__init__()
        d_inner = expand * d_model
        self.in_proj = nn.Linear(d_model, d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(d_inner, d_inner, kernel_size=d_conv, padding=d_conv - 1, groups=d_inner, bias=True)
        self.mixer = nn.Sequential(nn.Linear(d_inner, d_inner), nn.GELU(), nn.Linear(d_inner, d_inner))
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        L = x.shape[1]
        x_branch, z_branch = self.in_proj(self.norm(x)).chunk(2, dim=-1)
        x_conv = F.silu(self.conv1d(x_branch.transpose(1, 2))[:, :, :L].transpose(1, 2))
        return x + self.out_proj(self.mixer(x_conv) * F.silu(z_branch))


def build_fusion_blocks(d_model, num_blocks, d_state, d_conv, expand=MAMBA_EXPAND):
    if MAMBA_CLASS is not None:
        return nn.ModuleList([MambaBlock(d_model, d_state, d_conv, expand) for _ in range(num_blocks)])
    return nn.ModuleList([FallbackGatedConvBlock(d_model, d_conv, expand) for _ in range(num_blocks)])


# The reference results were produced by a model whose constructor drew, from the torch RNG, the initial weights
# of a 4-block (d_state 16, d_conv 4) Mamba stack before the final 2-block stack was built from the SAME starting
# RNG state. Every module built afterwards (pooling, classifier, auxiliary heads) and the whole training RNG stream
# (shuffle order, dropout masks) depend on the RNG state that 4-block draw leaves behind. To reproduce those
# results exactly, the constructor performs the same draw and discards it; the discarded blocks are never part of
# the model. This is a fixed step of weight initialization, not a configuration option.
RNG_REFERENCE_STACK = (4, 16, 4)   # (num_blocks, d_state, d_conv) whose initialization is drawn and discarded


class AeroSyncMamba(nn.Module):
    def __init__(self, d_model: int = D_MODEL, n_classes: int = N_CLASSES, htt_time: bool = True, laca_lag: bool = False):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes
        self.htt_time_enabled = htt_time                  # HTT_TIME_EMBED_ENABLED
        self.laca_lag_enabled = laca_lag                  # LACA_LAG_BIAS_ENABLED
        self.laca_beta = LACA_BETA if laca_lag else 0.0   # OFF = exactly 0.0 (v18's beta = 0 code path)

        self.sensor_encoder = nn.Sequential(
            nn.Conv1d(N_SENSOR_CHANNELS, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, d_model, kernel_size=3, padding=1), nn.BatchNorm1d(d_model), nn.GELU(),
        )
        self.audio_encoder = nn.Sequential(
            nn.Linear(N_AUDIO_BINS, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Linear(d_model, d_model)
        )
        self.htt = HeterogeneousTemporalTokenization(d_model, time_enabled=htt_time)
        self.laca = BiModalLACAFusion(d_model, beta=self.laca_beta)   # plain float: no params, no RNG draws
        self.sep_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        rng_before = torch.get_rng_state()
        build_fusion_blocks(d_model, *RNG_REFERENCE_STACK)
        rng_after_reference = torch.get_rng_state()
        torch.set_rng_state(rng_before)
        self.mamba_blocks = build_fusion_blocks(d_model, MAMBA_NUM_BLOCKS, MAMBA_D_STATE, MAMBA_D_CONV)
        torch.set_rng_state(rng_after_reference)

        self.pool_attn = nn.Sequential(nn.Linear(d_model, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, n_classes),
        )
        self.aux_a = nn.Linear(d_model, n_classes)
        self.aux_s = nn.Linear(d_model, n_classes)

    def forward(self, xa, xs, ta, ts):
        B = xa.shape[0]
        za = self.audio_encoder(xa)
        zs = self.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)

        Ha = self.htt(za, ta, modality_idx=0)
        Hs = self.htt(zs, ts, modality_idx=1)
        H_tilde_a, H_tilde_s, lags = self.laca(Ha, Hs, ta, ts)
        za_p = H_tilde_a.mean(dim=1)
        zs_p = H_tilde_s.mean(dim=1)

        F_seq = torch.cat([H_tilde_a, self.sep_token.expand(B, -1, -1), H_tilde_s], dim=1)
        for block in self.mamba_blocks:
            F_seq = block(F_seq)

        weights = F.softmax(self.pool_attn(F_seq), dim=1)
        zf = (F_seq * weights).sum(dim=1)
        logits = self.classifier(zf)

        pooled = {"z_a": za_p, "z_s": zs_p}
        aux_logits = {"y_a": self.aux_a(za_p), "y_s": self.aux_s(zs_p)}
        return logits, pooled, aux_logits, lags


def verify_architecture(model):
    """Finalized architecture, read from the constructed modules, including both v22 switches."""
    assert model.htt.time_enabled == model.htt_time_enabled
    beta = LACA_BETA if model.laca_lag_enabled else 0.0
    for name in ("laca_as", "laca_sa"):
        laca = getattr(model.laca, name)
        assert laca.num_heads == LACA_NUM_HEADS, f"{name}: {laca.num_heads} heads"
        assert laca.beta == beta, f"{name}: beta {laca.beta} != {beta} for LACA_LAG_BIAS_ENABLED={model.laca_lag_enabled}"
        assert sum(p.numel() for p in laca.lag_bias.parameters()) == LAG_BIAS_PARAMS
        assert isinstance(laca.lag_bias_norm, GridLayerNorm) and not list(laca.lag_bias_norm.parameters())
    assert len(model.mamba_blocks) == MAMBA_NUM_BLOCKS
    mamba_params = sum(p.numel() for p in model.mamba_blocks.parameters())
    if MAMBA_CLASS is not None:
        for block in model.mamba_blocks:
            m = block.mamba
            assert (m.d_state, m.d_conv, m.expand) == (MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND)
        assert mamba_params == EXPECTED_MAMBA_PARAMS, f"Mamba params {mamba_params:,} != {EXPECTED_MAMBA_PARAMS:,}"
    return mamba_params


_probe_counts = {}
for _c, (_h, _l) in CONDITIONS.items():
    _probe = AeroSyncMamba(htt_time=_h, laca_lag=_l)
    verify_architecture(_probe)
    assert _probe.htt.time_enabled == _h and _probe.laca.laca_as.beta == _probe.laca.laca_sa.beta == (LACA_BETA if _l else 0.0)
    _probe_counts[_c] = sum(p.numel() for p in _probe.parameters())
    print(f"Model (Condition {_c}): {_probe_counts[_c]:,} params | fusion blocks: {type(_probe.mamba_blocks[0]).__name__} x "
          f"{len(_probe.mamba_blocks)} ({verify_architecture(_probe):,} params) | HTT time embedding "
          f"{'ON' if _probe.htt.time_enabled else 'OFF'} | LACA beta {_probe.laca.laca_as.beta} / {_probe.laca.laca_sa.beta} | "
          f"backbone: {FUSION_BACKBONE}")
    del _probe
assert len(set(_probe_counts.values())) == 1, f"parameter count depends on the condition: {_probe_counts}"
print("[SWITCHES] HTT_TIME_EMBED_ENABLED -> HeterogeneousTemporalTokenization.forward: ON norm(z + e_m + time_embedding(tau)) "
      "| OFF norm(z + e_m)")
print(f"[SWITCHES] LACA_LAG_BIAS_ENABLED -> PairwiseLACA.forward: softmax(scores + beta * LN_grid(g(Δτ))) with beta = "
      f"{LACA_BETA} (ON, fixed, no warm-up) or exactly 0.0 (OFF, v18's beta = 0 path)")

## 5. Loss, metrics and the training function
The loss and optimizer are v22's, unchanged: class-weighted CE + 0.1 InfoNCE + 0.1 KL, AdamW 1e-4 with cosine decay,
20 epochs, and the best checkpoint by validation macro-F1.

`train_one_run(seed, condition)` is v24's function. It now takes the condition as an argument and trains on the
CORRECTED bundle only. Every RNG-consuming step happens in v24's order, so a retrained D follows v24's stream. The
switch checks are v25's: an OFF contribution must be exactly 0, and an ON one must be active.

In [ ]:
# ==============================================================================
# v26: v24's training cell, generalized from (seed, alignment) with Condition D fixed to (seed, condition) on
# CORRECTED_ALIGNMENT only. The loss, optimizer, schedule, checkpoint selection and the order of every RNG-consuming
# step are v24's, unchanged, so a retrained Condition D follows v24's RNG stream. The switch checks are v25's
# (they handle both ON and OFF).
# ==============================================================================
from sklearn.metrics import confusion_matrix


class AeroSyncLoss(nn.Module):
    """L = CE_classweighted(logits, y) + LAMBDA_ALIGN * InfoNCE(z_a, z_s) + LAMBDA_CONS * KL-consistency."""

    def __init__(self, class_weights):
        super().__init__()
        self.lambda_align = LAMBDA_ALIGN
        self.lambda_cons = LAMBDA_CONS
        self.temperature = INFONCE_TEMPERATURE
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def _infonce(self, z1, z2):
        sim = torch.matmul(F.normalize(z1, p=2, dim=-1), F.normalize(z2, p=2, dim=-1).t()) / self.temperature
        labels = torch.arange(z1.shape[0], device=z1.device)
        return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.t(), labels))

    def forward(self, logits, targets, pooled, aux_logits):
        loss_cls = self.ce(logits, targets)
        loss_align = self._infonce(pooled["z_a"], pooled["z_s"])
        p_fused = F.softmax(logits.detach(), dim=-1)
        loss_cons = (
            F.kl_div(F.log_softmax(aux_logits["y_a"], dim=-1), p_fused, reduction="batchmean") +
            F.kl_div(F.log_softmax(aux_logits["y_s"], dim=-1), p_fused, reduction="batchmean")
        ) / 2.0
        total = loss_cls + self.lambda_align * loss_align + self.lambda_cons * loss_cons
        return total, {"loss_cls": loss_cls.item(), "loss_align": loss_align.item(), "loss_cons": loss_cons.item()}


def compute_metrics(y_true, y_prob):
    """accuracy, macro-F1, balanced accuracy, macro one-vs-rest AUROC from class probabilities (prediction = argmax)."""
    y_pred = y_prob.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_auroc": float(roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")),
    }


def evaluate_probs(y_true, y_prob):
    """4 standard metrics + full 5x5 confusion matrix + per-class recall + Class 3/4 recall + 3->4 / 4->3 counts."""
    cm = confusion_matrix(y_true, y_prob.argmax(axis=1), labels=list(range(N_CLASSES)))
    recall = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)
    return {**compute_metrics(y_true, y_prob), "confusion_matrix": cm,
            **{f"recall_c{c}": float(recall[c]) for c in range(N_CLASSES)},
            "class3_recall": float(recall[3]), "class4_recall": float(recall[4]),
            "err_3to4": int(cm[3, 4]), "err_4to3": int(cm[4, 3])}


def print_confusion(ev, tag):
    cm = ev["confusion_matrix"]
    print(f"[CONFUSION {tag}] rows = TRUE severity, cols = PREDICTED severity (test set, {int(cm.sum())} windows)")
    print("            " + "".join(f"pred {j:>2} " for j in range(N_CLASSES)) + "|  recall")
    for i in range(N_CLASSES):
        print(f"   true {i:>2} " + "".join(f"{int(cm[i, j]):>8}" for j in range(N_CLASSES)) + f"  |  {ev[f'recall_c{i}']:.4f}")
    print(f"[CLASS 3/4 {tag}] Class 3 recall {ev['class3_recall']:.4f} | Class 4 recall {ev['class4_recall']:.4f} | "
          f"3->4 (true 3, pred 4) {ev['err_3to4']} | 4->3 (true 4, pred 3) {ev['err_4to3']}")


def train_class_weights(windows):
    """Inverse class frequency over the TRAINING windows."""
    counts = np.bincount([w["label"] for w in windows], minlength=N_CLASSES)
    return torch.tensor(counts.sum() / (len(counts) * np.maximum(counts, 1)), dtype=torch.float32)


def _batch_to_device(batch, non_blocking=False):
    return tuple(batch[k].to(device, non_blocking=non_blocking) for k in ("x_a", "x_s", "tau_a", "tau_s", "label"))


def predict_probs(model, loader):
    """Softmax probabilities and true labels over a loader (eval mode, no grad), in loader order."""
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            xa, xs, ta, ts, y = _batch_to_device(batch)
            logits, _, _, _ = model(xa, xs, ta, ts)
            probs.append(F.softmax(logits, dim=-1).cpu().numpy())
            labels.append(y.cpu().numpy())
    return np.concatenate(probs), np.concatenate(labels)


METRICS = ["accuracy", "macro_f1", "balanced_accuracy", "macro_auroc"]


def make_bundle(data):
    """Loaders, class weights and probe batches for one dataset — built exactly as v24 built them."""
    tr = data["train_ds"]
    return {**data,
            "train_loader": DataLoader(tr, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, num_workers=NUM_WORKERS),
            "val_loader": DataLoader(data["val_ds"], batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS),
            "test_loader": DataLoader(data["test_ds"], batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS),
            "class_weights": train_class_weights(data["split_windows"]["train"]).to(device),
            "test_labels": np.array([w["label"] for w in data["split_windows"]["test"]], dtype=np.int64),
            "test_missions": np.array(data["window_missions"]["test"]),
            # fixed probe batches, read by index (no DataLoader, no RNG)
            "lag_probe_taus": tuple(torch.stack([tr[i][k] for i in range(min(BATCH_SIZE, len(tr)))]) for k in ("tau_a", "tau_s")),
            "switch_probe_idx": np.linspace(0, len(tr) - 1, BATCH_SIZE).astype(int)}


BUNDLE = make_bundle(DATASETS[ALIGNMENT])
TEST_LABELS = BUNDLE["test_labels"]
assert len(BUNDLE["test_labels"]) == len(BUNDLE["test_missions"]) == len(BUNDLE["test_ds"])
print(f"[BUNDLE] {ALIGNMENT}: train {len(BUNDLE['train_ds'])} / val {len(BUNDLE['val_ds'])} / test {len(BUNDLE['test_ds'])} "
      f"windows | class weights {np.round(BUNDLE['class_weights'].cpu().numpy(), 4).tolist()}")


def lag_bias_contribution(model, bundle):
    """Per LACA direction: max|beta * LN_grid(g(Δτ))|, the exact term PairwiseLACA adds to the attention scores."""
    ta, ts = (t.to(device) for t in bundle["lag_probe_taus"])
    out = {}
    with torch.no_grad():
        for name, laca, tau_m, tau_n in (("laca_as", model.laca.laca_as, ta, ts), ("laca_sa", model.laca.laca_sa, ts, ta)):
            contrib = laca.beta * laca.lag_bias_term(tau_m, tau_n)
            out[name] = {"beta": laca.beta, "contrib_absmax": float(contrib.abs().max()),
                         "contrib_finite": bool(torch.isfinite(contrib).all())}
    return out


def htt_time_contribution(model, bundle):
    """What HTT's time embedding contributes on a fixed real batch (eval mode, RNG untouched):
    term = max|HTT(z, tau) - norm(z + e_m)| and tau_sens = max|HTT(z, tau) - HTT(z, tau + 0.3)|, per modality."""
    items = [bundle["train_ds"][int(i)] for i in bundle["switch_probe_idx"]]
    xa, xs, ta, ts = (torch.stack([it[k] for it in items]).to(device) for k in ("x_a", "x_s", "tau_a", "tau_s"))
    was_training = model.training
    cuda_devices = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []
    out = {}
    with torch.random.fork_rng(devices=cuda_devices), torch.no_grad():
        model.eval()
        za = model.audio_encoder(xa)
        zs = model.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)
        for name, z, tau, idx in (("audio", za, ta, 0), ("sensor", zs, ts, 1)):
            e_m = model.htt.modality_embeddings(torch.tensor(idx, device=device)).view(1, 1, model.d_model)
            actual = model.htt(z, tau, modality_idx=idx)
            out[name] = {"term": float((actual - model.htt.norm(z + e_m)).abs().max()),
                         "tau_sens": float((actual - model.htt(z, tau + 0.3, modality_idx=idx)).abs().max())}
    model.train(was_training)
    return out


def check_switches(model, bundle, tag, when, final=False):
    """v25's check: a disabled contribution must be EXACTLY 0; an enabled one must be active (after training)."""
    h = htt_time_contribution(model, bundle)
    lag = lag_bias_contribution(model, bundle)
    h_max = max(max(v["term"], v["tau_sens"]) for v in h.values())
    lag_max = max(r["contrib_absmax"] for r in lag.values())
    if not model.htt_time_enabled:
        assert h_max == 0.0, f"HTT time embedding is OFF but contributes {h_max!r}"
    else:
        assert h_max > 0.0, "HTT time embedding is ON but contributes nothing"
    if not model.laca_lag_enabled:
        assert all(r["contrib_finite"] and r["contrib_absmax"] == 0.0 for r in lag.values()), f"LACA lag OFF but {lag}"
    lag_msg = ("EXACTLY 0 (OFF)" if not model.laca_lag_enabled else
               f"{lag_max:.4f} (ON" + (", 0 only because g's final layer is zero-initialized (v18 form)" if lag_max == 0.0 and not final
                                     else "") + ")")
    print(f"{tag} [SWITCH CHECK {when}] HTT time term {'EXACTLY 0 (OFF)' if not model.htt_time_enabled else f'{h_max:.3e} (ON)'} "
          f"| LACA lag term {lag_msg} ✅")
    if final and model.laca_lag_enabled and lag_max == 0.0:
        print(f"🚨 {tag} LACA lag is ON but its contribution is still 0 after training — the switch is not working")
    return h_max, lag_max


def train_one_run(seed: int, cond: str):
    """Train Condition `cond` with `seed` on CORRECTED_ALIGNMENT, evaluate on its test set, save the best checkpoint.
    Returns (trained model, result dict incl. test probabilities and confusion matrix)."""
    t0 = time.time()
    bundle = BUNDLE
    htt_on, laca_on = CONDITIONS[cond]
    tag = f"[{ALIGNMENT} | {cond} seed={seed}]"
    set_all_seeds(seed)
    model = AeroSyncMamba(htt_time=htt_on, laca_lag=laca_on).to(device)
    init_sha1 = hashlib.sha1(b"".join(v.detach().cpu().contiguous().numpy().tobytes()
                                      for v in model.state_dict().values())).hexdigest()
    rng_sha1 = hashlib.sha1(torch.get_rng_state().numpy().tobytes()).hexdigest()
    mamba_params = verify_architecture(model)
    ref_init, ref_rng = SEED_INIT_REFERENCE[seed]
    paired_ok = init_sha1[:12] == ref_init and rng_sha1[:12] == ref_rng
    print(f"{tag} HTT_TIME_EMBED_ENABLED = {model.htt.time_enabled} | LACA_LAG_BIAS_ENABLED = {model.laca_lag_enabled} "
          f"(beta {model.laca.laca_as.beta}) | fusion {type(model.mamba_blocks[0]).__name__} ({mamba_params:,} params) | "
          f"train {len(bundle['train_ds'])} / val {len(bundle['val_ds'])} / test {len(bundle['test_ds'])} windows")
    print(f"{tag} [PAIRING] initial weights {init_sha1[:12]} vs reference {ref_init} | RNG {rng_sha1[:12]} vs reference "
          f"{ref_rng} -> " + ("✅ identical start (same for C and D of this seed)" if paired_ok else
                              "🚨 MISMATCH — this seed's conditions are not cleanly paired"))
    check_switches(model, bundle, tag, "before training")

    criterion = AeroSyncLoss(bundle["class_weights"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)

    best_val_f1, best_state, best_epoch = 0.0, None, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = defaultdict(float)
        for batch in tqdm(bundle["train_loader"], desc=f"{cond} seed {seed} epoch {epoch:02d}/{EPOCHS}", leave=False):
            xa, xs, ta, ts, y = _batch_to_device(batch, non_blocking=True)
            optimizer.zero_grad()
            logits, pooled, aux_logits, _ = model(xa, xs, ta, ts)
            loss, parts = criterion(logits, y, pooled, aux_logits)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()
            running["loss"] += loss.item()
            for k, v in parts.items():
                running[k] += v
        scheduler.step()

        lag = lag_bias_contribution(model, bundle)
        lag_max = max(r["contrib_absmax"] for r in lag.values())
        if not laca_on:
            assert all(r["contrib_finite"] and r["contrib_absmax"] == 0.0 for r in lag.values()), f"lag leak at epoch {epoch}"

        val_probs, val_labels = predict_probs(model, bundle["val_loader"])
        val_f1 = compute_metrics(val_labels, val_probs)["macro_f1"]
        improved = val_f1 > best_val_f1
        if improved:
            best_val_f1, best_state, best_epoch = val_f1, copy.deepcopy(model.state_dict()), epoch
        n = len(bundle["train_loader"])
        print(f"{tag} epoch {epoch:02d}/{EPOCHS} | lag term {lag_max:.4f}{' (exactly 0 ✅)' if not laca_on else ''} | train "
              f"loss {running['loss'] / n:.4f} (cls {running['loss_cls'] / n:.4f}, align {running['loss_align'] / n:.4f}, "
              f"cons {running['loss_cons'] / n:.4f}) | val macro-F1 {val_f1:.4f}{'  * best' if improved else ''}")

    model.load_state_dict(best_state)
    check_switches(model, bundle, tag, f"best checkpoint, epoch {best_epoch}", final=True)
    test_probs, test_labels = predict_probs(model, bundle["test_loader"])
    assert np.array_equal(test_labels, bundle["test_labels"])
    ev = evaluate_probs(test_labels, test_probs)
    run_name = f"{ALIGNMENT}_{cond}_seed{seed}"
    ckpt = CHECKPOINT_DIR / f"{run_name}_best.pt"
    torch.save(best_state, ckpt)
    np.save(PROBS_DIR / f"{run_name}_test_probs.npy", test_probs)
    np.save(PROBS_DIR / f"{run_name}_test_labels.npy", test_labels)

    result = {
        "alignment": ALIGNMENT, "condition": cond, "seed": seed,
        **ev,
        "n_test_windows": len(test_labels),
        "best_val_macro_f1": round(best_val_f1, 6),
        "best_epoch": best_epoch,
        "init_sha1": init_sha1[:12],
        "rng_sha1": rng_sha1[:12],
        "paired_init": paired_ok,
        "checkpoint": str(ckpt),
        "train_seconds": round(time.time() - t0, 1),
        "test_probs": test_probs,
        "test_labels": test_labels,
    }
    return model, result

## 6. Step 1 — baseline models at 0 ms (aligned data)
For each seed, D first, then C:
- **D** loads `CORRECTED_ALIGNMENT_D_seed{s}_best.pt` if one is attached, from `v24_checkpoints/` or an earlier
  `v26_checkpoints/`, and evaluates it on the CORRECTED test set. It is **reused only if its macro-F1 is within ±0.0005
  of v24's**. Otherwise it is retrained.
- **C** is always trained.

Every model ends up in `/kaggle/working/v26_checkpoints/`, and GPU memory is freed after each run. The D reproduction
check (PASS/FAIL per seed) and the C sanity check are printed at the end of this cell, before Step 2.

> **Keep the checkpoints:** save this run as a notebook version **with output**, then attach it as an input next time.

In [ ]:
# ==============================================================================
# STEP 1 — BASELINE MODELS at 0 ms (aligned data). D: reuse v24's checkpoint if it reproduces v24, else retrain.
# C: always trained (a NEW baseline; there is nothing to reproduce). Checkpoints -> CHECKPOINT_DIR. GPU freed per run.
# ==============================================================================
for _f in (RESULTS_CSV, TRAINING_CSV):
    if _f.exists():
        _f.rename(_f.with_suffix(f".{datetime.now():%Y%m%d_%H%M%S}.bak.csv"))
TRAINING_FIELDS = (["condition", "seed", "source", "checkpoint"] + METRICS + ["class3_recall", "class4_recall",
                   "confusion_matrix", "n_test_windows", "reference", "reference_macro_f1", "delta_macro_f1",
                   "max_abs_delta_4_metrics", "confusion_identical", "check", "best_epoch", "best_val_macro_f1",
                   "init_sha1", "rng_sha1", "paired_init", "train_seconds", "finished_at"])


def _append(path, fields, row):
    write_header = not path.exists()
    with open(path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        if write_header:
            w.writeheader()
        w.writerow({k: row.get(k) for k in fields})


def _find_d_checkpoint(seed):
    name = f"{ALIGNMENT}_D_seed{seed}_best.pt"
    for pattern in D_CHECKPOINT_GLOBS:
        hits = sorted(glob.glob(pattern.format(name=name), recursive=True))
        if hits:
            return hits[0]
    return None


def _free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _d_vs_v24(seed, ev):
    ref = V24_CORRECTED_D[seed]
    d_f1 = ev["macro_f1"] - ref["macro_f1"]
    worst = max(abs(ev[m] - ref[m]) for m in METRICS)
    cm_same = bool(np.array_equal(np.asarray(ev["confusion_matrix"]), np.array(ref["confusion_matrix"])))
    return d_f1, worst, cm_same, abs(d_f1) <= SAME_SEED_TOLERANCE


BASELINES = {}          # (cond, seed) -> result dict (test probs on the full CORRECTED test set, metrics, checkpoint path)
MODEL_SOURCES = {}      # (cond, seed) -> "reused checkpoint" | "trained in v26"
D_REPRO = {}            # seed -> (delta macro-F1, max |delta| over 4 metrics, confusion identical, PASS)
C_SANITY = {}           # seed -> (delta vs v25 OLD-alignment C, within C_SANITY_MAX_ABS)
n_trained, t_step1 = 0, time.time()
TRAIN_TIMES = []

for slot, (seed, cond) in enumerate(RUN_ORDER, start=1):
    print("\n" + "=" * 100 + f"\nSTEP 1 — MODEL {slot}/{len(RUN_ORDER)}: Condition {cond} (HTT-time "
          f"{'ON' if CONDITIONS[cond][0] else 'OFF'}, LACA-lag {'ON' if CONDITIONS[cond][1] else 'OFF'}), seed {seed}\n" + "=" * 100)
    tag = f"[{cond} seed={seed}]"
    result, model = None, None
    if cond == "D":
        path = _find_d_checkpoint(seed)
        if path is not None:
            model = AeroSyncMamba(htt_time=False, laca_lag=False).to(device)
            model.load_state_dict(torch.load(path, map_location=device))
            verify_architecture(model)
            probs, labels = predict_probs(model, BUNDLE["test_loader"])
            assert np.array_equal(labels, TEST_LABELS)
            ev = evaluate_probs(labels, probs)
            d_f1, worst, cm_same, ok = _d_vs_v24(seed, ev)
            print(f"{tag} found checkpoint {path}: macro-F1 {ev['macro_f1']:.4f} vs v24 {V24_CORRECTED_D[seed]['macro_f1']:.4f} "
                  f"({d_f1:+.1e}) | confusion matrix {'identical' if cm_same else 'DIFFERENT'}")
            if ok:
                ckpt = CHECKPOINT_DIR / f"{ALIGNMENT}_D_seed{seed}_best.pt"
                if Path(path).resolve() != ckpt.resolve():
                    shutil.copyfile(path, ckpt)
                np.save(PROBS_DIR / f"{ALIGNMENT}_D_seed{seed}_test_probs.npy", probs)
                result = {"condition": "D", "seed": seed, **ev, "test_probs": probs, "test_labels": labels,
                          "n_test_windows": len(labels), "checkpoint": str(ckpt), "source_checkpoint": path,
                          "best_epoch": None, "paired_init": None}
                MODEL_SOURCES[(cond, seed)] = "reused checkpoint"
                print(f"{tag} ✅ REUSED checkpoint (reproduces v24; not retrained); copied to {ckpt}")
            else:
                print(f"{tag} 🚨 checkpoint does NOT reproduce v24 (|Δ macro-F1| > {SAME_SEED_TOLERANCE}) -> retraining")
                del model
                model = None
                _free_gpu()
        else:
            print(f"{tag} no v24 checkpoint found (searched {D_CHECKPOINT_GLOBS[0].format(name=f'{ALIGNMENT}_D_seed{seed}_best.pt')} "
                  f"and {len(D_CHECKPOINT_GLOBS) - 1} other locations) -> RETRAINING (~2-4 min)")
    if result is None:
        model, result = train_one_run(seed, cond)
        MODEL_SOURCES[(cond, seed)] = "trained in v26"
        n_trained += 1
        TRAIN_TIMES.append(result["train_seconds"])
    BASELINES[(cond, seed)] = result
    print(f"{tag} ({MODEL_SOURCES[(cond, seed)]}) TEST (full CORRECTED test set, {result['n_test_windows']} windows): "
          f"accuracy {result['accuracy']:.4f} | macro-F1 {result['macro_f1']:.4f} | balanced accuracy "
          f"{result['balanced_accuracy']:.4f} | macro-AUROC {result['macro_auroc']:.4f}")
    print_confusion(result, f"{cond} seed={seed}")

    row = {**{k: result.get(k) for k in TRAINING_FIELDS}, "condition": cond, "seed": seed, "source": MODEL_SOURCES[(cond, seed)],
           "confusion_matrix": json.dumps(np.asarray(result["confusion_matrix"]).tolist()),
           "finished_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
    if cond == "D":
        D_REPRO[seed] = _d_vs_v24(seed, result)
        d_f1, worst, cm_same, ok = D_REPRO[seed]
        row.update(reference="v24 CORRECTED D", reference_macro_f1=V24_CORRECTED_D[seed]["macro_f1"], delta_macro_f1=d_f1,
                   max_abs_delta_4_metrics=worst, confusion_identical=cm_same, check="PASS" if ok else "FAIL")
    else:
        ref = V25_OLD_ALIGNMENT_C_F1[seed]
        C_SANITY[seed] = (result["macro_f1"] - ref, abs(result["macro_f1"] - ref) <= C_SANITY_MAX_ABS)
        row.update(reference="v25 OLD-alignment C (context only)", reference_macro_f1=ref, delta_macro_f1=C_SANITY[seed][0],
                   check="within range" if C_SANITY[seed][1] else "WARN")
    _append(TRAINING_CSV, TRAINING_FIELDS, row)

    if model is not None:
        del model
    _free_gpu()
    print(f">>> {slot}/{len(RUN_ORDER)} models done | {n_trained} trained | elapsed {(time.time() - t_step1) / 60:.1f} min")

# ------------------------------------------------------------------------------
# STEP 1 CHECKS — printed before Step 2
# ------------------------------------------------------------------------------
print("\n" + "█" * 110)
print("█  STEP 1 CHECK A — Condition D REPRODUCTION vs v24 (CORRECTED_ALIGNMENT, full test set, tolerance ±"
      f"{SAME_SEED_TOLERANCE} macro-F1)")
print("█" * 110)
for seed in SEEDS:
    d_f1, worst, cm_same, ok = D_REPRO[seed]
    print(f"█  seed {seed:<5} {MODEL_SOURCES[('D', seed)]:<22} macro-F1 {BASELINES[('D', seed)]['macro_f1']:.4f} vs v24 "
          f"{V24_CORRECTED_D[seed]['macro_f1']:.4f} (Δ {d_f1:+.5f}) | max |Δ| 4 metrics {worst:.1e} | CM "
          f"{'identical' if cm_same else 'DIFFERENT'} -> {'✅ PASS' if ok else '🚨 FAIL'}")
D_REPRO_ALL_PASS = all(v[3] for v in D_REPRO.values())
if D_REPRO_ALL_PASS:
    print("█  ✅ ALL D SEEDS REPRODUCE v24.")
else:
    print("█  " + "🚨" * 30)
    print("█  🚨 D DOES NOT REPRODUCE v24 FOR AT LEAST ONE SEED. Step 2 still runs: every drop is measured against the SAME")
    print("█  🚨 model's own 0 ms score, so the manipulation check stays internally valid, but these D models are not v24's.")
    print("█  " + "🚨" * 30)
print("█" * 110)
print("█  STEP 1 CHECK B — Condition C SANITY (a NEW baseline: no CORRECTED-alignment C was ever trained; NOT a reproduction)")
print("█  Reference = v25's OLD-alignment C (99 missions, 1130 test windows) — a different dataset, shown only as a range.")
print("█" * 110)
for seed in SEEDS:
    delta, within = C_SANITY[seed]
    print(f"█  seed {seed:<5} C (CORRECTED) macro-F1 {BASELINES[('C', seed)]['macro_f1']:.4f} | v25 OLD-alignment C "
          f"{V25_OLD_ALIGNMENT_C_F1[seed]:.4f} | Δ {delta:+.4f} -> " + ("within ±0.03 ✅" if within else
                                                                       f"⚠️ WARNING: more than {C_SANITY_MAX_ABS} away"))
    print(f"█             D (CORRECTED) macro-F1 {BASELINES[('D', seed)]['macro_f1']:.4f} -> C - D at 0 ms "
          f"{BASELINES[('C', seed)]['macro_f1'] - BASELINES[('D', seed)]['macro_f1']:+.4f}")
print("█" * 110)
print(f"\nStep 1 done in {(time.time() - t_step1) / 60:.1f} min: {n_trained} trained "
      + (f"(mean {np.mean(TRAIN_TIMES) / 60:.1f} min per run)" if TRAIN_TIMES else "") + f", "
      f"{len(RUN_ORDER) - n_trained} reused.")
print("\n" + "!" * 110)
print(f"!  REMINDER: every checkpoint is in {CHECKPOINT_DIR} ({len(list(CHECKPOINT_DIR.glob('*.pt')))} files). Kaggle keeps them")
print("!  ONLY if this run is saved as a notebook version with its OUTPUT (Save Version -> Save & Run All / Quick Save with")
print("!  output). Attach that output as an input next time to reuse the models instead of retraining.")
print("!" * 110)

## 7. Step 2a — real content-shift injection and its verification hooks
The injection is v25's `content_shift` code (`_shift_valid` and the sensor half of `_window_content_shift`), ported to
the CORRECTED windows. Each stored test window keeps its audio (`x_a`, `τ_a`) exactly. Its sensor window is re-sliced
from the raw IMU log over `[t_start + offset, t_start + W + offset]`.
- **Honest τ_s** = the true time of the shifted samples minus the **audio** window start, divided by W. At +1 s this is
  about 0.39 to 1.39.
- **Re-zeroed τ_s** = the same samples measured from `t_start + offset`, divided by W: the normal 0 to 1 range.

Hooks, in order:
1. **Re-slice at offset 0** must equal the stored standardized test tensors: max |diff| and PASS/FAIL. **On FAIL the
   cell prints a loud warning and raises, so Step 2 does not run.** Step 1's checkpoints are already saved, and a re-run
   in the same session reuses the D ones.
2. **Retention per offset**, with the reasons for each drop, and the size of the all-offset **matched set**. The raw IMU
   stream starts at t = 0, about 19 s before the audio onset, so a window is never dropped just because it precedes the
   onset.
3. **Sample windows** at each nonzero offset: the sensor range retrieved vs the audio window. Then the **achieved
   offset** per configuration, mean over matched windows, against the requested one.
4. **A concrete τ example** at ±1.0 s: τ_a, honest τ_s and re-zeroed τ_s. It confirms that the honest mean gap moves by
   offset/W and the re-zeroed gap does not.

In [ ]:
# ==============================================================================
# STEP 2a — REAL CONTENT-SHIFT INJECTION (evaluation only), ported from v25's content_shift arm to the CORRECTED windows.
# For each test window: the stored audio window [t_start, t_start + W] is kept as is (x_a, tau_a unchanged). The sensor
# window is re-sliced from the RAW IMU log over [t_start + offset, t_start + W + offset] with v18/v23's own sensor
# slicing (TS_SENSOR real samples chosen evenly by index, same np.interp), then standardized with the CORRECTED
# TRAINING-set mean/std in the same channel order (UAVWindowDataset). tau is normalized by W exactly as the pipeline does.
#   honest   tau_s = (true time of the shifted samples - AUDIO window start) / W   -> about [offset/W, 1 + offset/W]
#   rezeroed tau_s = (true time of the shifted samples - (t_start + offset)) / W  -> the normal [0, 1] range
# ==============================================================================
def stack_dataset(ds):
    """All windows of a UAVWindowDataset as stacked tensors, in index order (= the unshuffled test_loader order)."""
    items = [ds[i] for i in range(len(ds))]
    return {k: torch.stack([it[k] for it in items]) for k in ("x_a", "x_s", "tau_a", "tau_s", "label")}


def _corrected_window_starts(raw):
    """t_start of every window slice_windows_CORRECTED_ALIGNMENT returns for this mission, in order (same start rule,
    same += STEP_SEC accumulation, same end rule, same sensor-count skip). Builds no window."""
    s_times, a_times = raw["sensor_times"], raw["audio_times"]
    max_time = min(s_times[-1], a_times[-1])
    starts = []
    t_start = max(0.0, float(a_times[0])) if CORRECTED_WINDOW_START == "audio_onset" else 0.0
    while t_start + WINDOW_SEC <= max_time:
        if (((s_times >= t_start) & (s_times <= t_start + WINDOW_SEC)).sum()) >= MIN_SENSOR_SAMPLES_PER_WINDOW:
            starts.append(t_start)
        t_start += STEP_SEC
    return starts


def _shift_status(raw, t_start, offset_s):
    """v25's validity rule on the RAW IMU stream (which starts at t = 0, well before the audio onset): 'ok', or why not.
    A window is NOT dropped for preceding the audio onset — only for leaving the recorded IMU stream."""
    s_times = raw["sensor_times"]
    lo, hi = t_start + offset_s, t_start + WINDOW_SEC + offset_s
    if lo < 0.0:
        return "before IMU start"
    if hi > s_times[-1]:
        return "after IMU end"
    if int(((s_times >= lo) & (s_times <= hi)).sum()) < MIN_SENSOR_SAMPLES_PER_WINDOW:
        return "too few IMU samples"
    return "ok"


def reslice_sensor(raw, t_start, offset_s):
    """v25's _window_content_shift, sensor part (= slice_windows' sensor code with the window moved by offset_s).
    Returns raw x_s (TS_SENSOR, 6) and the ABSOLUTE times (s from the first IMU sample) of the selected samples."""
    s_times, telemetry = raw["sensor_times"], raw["telemetry"]
    t_end = t_start + WINDOW_SEC
    s_mask = (s_times >= t_start + offset_s) & (s_times <= t_end + offset_s)
    s_idx = np.where(s_mask)[0]
    tau_s_abs = s_times[s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]]
    x_s = np.zeros((TS_SENSOR, N_SENSOR_CHANNELS), dtype=np.float32)
    for c in range(N_SENSOR_CHANNELS):
        x_s[:, c] = np.interp(tau_s_abs, s_times[s_mask], telemetry[s_mask, c])
    return x_s, tau_s_abs


# --- index of every stored CORRECTED test window: (mission, t_start), in corr_windows["test"] order ---
TEST_WINDOWS = CORRECTED_ALIGNMENT["split_windows"]["test"]
TEST_WIN_MISSIONS = list(CORRECTED_ALIGNMENT["window_missions"]["test"])
N_TEST = len(TEST_WINDOWS)
TEST_INDEX = []
for _name in dict.fromkeys(TEST_WIN_MISSIONS):          # missions in window order, each once
    _st = _corrected_window_starts(CORRECTED_TEST_IMU[_name])
    assert len(_st) == TEST_WIN_MISSIONS.count(_name), f"{_name}: {len(_st)} recomputed starts vs {TEST_WIN_MISSIONS.count(_name)} windows"
    TEST_INDEX.extend((_name, t) for t in _st)
assert len(TEST_INDEX) == N_TEST and [m for m, _ in TEST_INDEX] == TEST_WIN_MISSIONS
FULL0 = stack_dataset(BUNDLE["test_ds"])                 # the stored, normally preprocessed test tensors
assert np.array_equal(FULL0["label"].numpy(), TEST_LABELS)
SENSOR_MEAN, SENSOR_STD = BUNDLE["train_ds"].sensor_mean, BUNDLE["train_ds"].sensor_std   # CORRECTED TRAINING stats


def build_inputs(indices, offset_s, tau_mode):
    """Stacked model inputs for the given test-window indices at `offset_s`. Audio (x_a, tau_a) and the label are the
    stored window's; x_s is re-sliced from the raw IMU log; tau_s follows `tau_mode`."""
    wins = []
    for i in indices:
        name, t_start = TEST_INDEX[i]
        x_s, tau_abs = reslice_sensor(CORRECTED_TEST_IMU[name], t_start, offset_s)
        ref = t_start if tau_mode == "honest" else t_start + offset_s
        wins.append({**TEST_WINDOWS[i], "x_s": x_s, "tau_s": (tau_abs - ref).astype(np.float32)})
    return stack_dataset(UAVWindowDataset(wins, SENSOR_MEAN, SENSOR_STD))


# ------------------------------------------------------------------------------
# HOOK 1 — offset 0 re-sliced from the RAW IMU log must reproduce the stored preprocessed test windows
# ------------------------------------------------------------------------------
_r0 = build_inputs(range(N_TEST), 0.0, "honest")
RESLICE_MAXDIFF = {k: float((_r0[k].double() - FULL0[k].double()).abs().max()) for k in ("x_s", "tau_s", "x_a", "tau_a", "label")}
RESLICE_BIT_IDENTICAL = all(torch.equal(_r0[k], FULL0[k]) for k in FULL0)
RESLICE_PASS = max(RESLICE_MAXDIFF.values()) <= RESLICE_TOLERANCE
print("█" * 110)
print(f"█  HOOK 1 — RE-SLICE CHECK at offset 0 ({N_TEST} test windows, raw IMU log -> standardized tensors vs stored)")
print("█    max |difference|: " + ", ".join(f"{k} {v:.3e}" for k, v in RESLICE_MAXDIFF.items())
      + f" | bit-identical: {RESLICE_BIT_IDENTICAL} | tolerance {RESLICE_TOLERANCE:.0e} -> "
      + ("✅ PASS" if RESLICE_PASS else "🚨 FAIL"))
print("█" * 110)
del _r0
if not RESLICE_PASS:
    print("\n" + "🚨" * 50)
    print("🚨 THE RE-SLICER DOES NOT REPRODUCE THE STORED TEST WINDOWS AT OFFSET 0.")
    print("🚨 Any 'drop' measured in Step 2 would include a preprocessing difference, not only misalignment.")
    print("🚨 STOPPING BEFORE STEP 2. Step 1's checkpoints are saved and will be reused by a re-run in this session.")
    print("🚨" * 50)
    raise RuntimeError("v26 offset-0 re-slice check FAILED — Step 2 not run (see the warning above)")

# ------------------------------------------------------------------------------
# HOOK 2 — retention per offset (drop only if the shifted window leaves the raw IMU stream) and the matched set
# ------------------------------------------------------------------------------
STATUS = {o: [_shift_status(CORRECTED_TEST_IMU[m], t, o) for m, t in TEST_INDEX] for o in OFFSETS_S}
VALID = {o: np.array([s == "ok" for s in STATUS[o]]) for o in OFFSETS_S}
assert VALID[0.0].all(), "a stored test window is invalid at offset 0"
MATCHED_MASK = np.logical_and.reduce([VALID[o] for o in OFFSETS_S])
MATCHED_IDX = np.where(MATCHED_MASK)[0]
MATCHED_LABELS = TEST_LABELS[MATCHED_IDX]
_full_cc = np.bincount(TEST_LABELS, minlength=N_CLASSES)
print(f"\n[RETENTION] test windows kept per offset (of {N_TEST}; full test per class {_full_cc.tolist()})")
print(f"    {'offset':>8}{'kept':>7}{'lost':>6}{'lost %':>9}   reasons{'':<34}kept per class")
RETENTION = {}
for o in OFFSETS_S:
    _reasons = defaultdict(int)
    for s in STATUS[o]:
        if s != "ok":
            _reasons[s] += 1
    kept = int(VALID[o].sum())
    RETENTION[o] = {"kept": kept, "lost": N_TEST - kept, "lost_pct": 100.0 * (N_TEST - kept) / N_TEST, "reasons": dict(_reasons)}
    print(f"    {o:>+8.1f}{kept:>7}{N_TEST - kept:>6}{RETENTION[o]['lost_pct']:>8.2f}%   {str(dict(_reasons)) or '{}':<41}"
          f"{np.bincount(TEST_LABELS[VALID[o]], minlength=N_CLASSES).tolist()}")
print(f"[MATCHED SET] windows valid at ALL {len(OFFSETS_S)} offsets: {len(MATCHED_IDX)} of {N_TEST} "
      f"({100.0 * len(MATCHED_IDX) / N_TEST:.2f}%) | per class {np.bincount(MATCHED_LABELS, minlength=N_CLASSES).tolist()} "
      f"| missions {len({TEST_INDEX[i][0] for i in MATCHED_IDX})} of {len(set(TEST_WIN_MISSIONS))}")
print("[MATCHED SET] every offset and every model is scored on these SAME windows, so the drops are directly comparable. "
      "The 0 ms score on the full test set is also logged for reference.")

# ------------------------------------------------------------------------------
# HOOK 3 — sample windows: sensor time range actually retrieved vs the audio window; achieved offset per configuration
# ------------------------------------------------------------------------------
_abs0 = {i: reslice_sensor(CORRECTED_TEST_IMU[TEST_INDEX[i][0]], TEST_INDEX[i][1], 0.0)[1] for i in MATCHED_IDX}
ACHIEVED = {}
_sample_pos = np.linspace(0, len(MATCHED_IDX) - 1, N_SAMPLE_WINDOWS).round().astype(int)
for o in OFFSETS_S:
    diffs = np.array([reslice_sensor(CORRECTED_TEST_IMU[TEST_INDEX[i][0]], TEST_INDEX[i][1], o)[1].mean() - _abs0[i].mean()
                      for i in MATCHED_IDX])
    ACHIEVED[o] = {"mean": float(diffs.mean()), "min": float(diffs.min()), "max": float(diffs.max())}
    if o == 0.0:
        continue
    print(f"\n[SAMPLES offset {o:+.1f} s] ({'IMU delayed: sensor data from LATER' if o > 0 else 'audio delayed: sensor data from EARLIER'})")
    for p in _sample_pos:
        i = MATCHED_IDX[p]
        name, t = TEST_INDEX[i]
        tau_a_abs = t + TEST_WINDOWS[i]["tau_a"].astype(np.float64)
        s_abs = reslice_sensor(CORRECTED_TEST_IMU[name], t, o)[1]
        print(f"    {name} (class {TEST_WINDOWS[i]['label']}) | audio window [{t:9.3f}, {t + WINDOW_SEC:9.3f}] s, audio chunk "
              f"times [{tau_a_abs.min():9.3f}, {tau_a_abs.max():9.3f}] | sensor retrieved [{s_abs.min():9.3f}, "
              f"{s_abs.max():9.3f}] (unshifted [{_abs0[i].min():9.3f}, {_abs0[i].max():9.3f}]) | achieved "
              f"{s_abs.mean() - _abs0[i].mean():+.4f} s")
print(f"\n[ACHIEVED OFFSET] mean over the {len(MATCHED_IDX)} matched windows of (mean shifted sensor time - mean unshifted "
      f"sensor time), per configuration:")
for o in OFFSETS_S:
    a = ACHIEVED[o]
    print(f"    requested {o:+.2f} s -> achieved mean {a['mean']:+.4f} s (min {a['min']:+.4f}, max {a['max']:+.4f}) | "
          f"|mean - requested| {abs(a['mean'] - o) * 1e3:.2f} ms")
ACHIEVED_OK = all(abs(ACHIEVED[o]["mean"] - o) < 0.05 for o in OFFSETS_S)
print("    " + ("✅ every configuration achieves its requested offset within 50 ms on average" if ACHIEVED_OK else
                "🚨 an achieved offset is more than 50 ms from the requested one"))
del _abs0

# ------------------------------------------------------------------------------
# Build the matched-set inputs: every offset x every tau mode (audio identical across all of them)
# ------------------------------------------------------------------------------
MATCHED_INPUTS = {(o, m): build_inputs(MATCHED_IDX, o, m) for o in OFFSETS_S for m in TAU_MODES}
_m0 = {k: v[MATCHED_IDX] for k, v in FULL0.items()}
assert all(float((MATCHED_INPUTS[(0.0, m)][k].double() - _m0[k].double()).abs().max()) <= RESLICE_TOLERANCE
           for m in TAU_MODES for k in _m0), "matched-set inputs at offset 0 differ from the stored test windows"
for (o, m), inp in MATCHED_INPUTS.items():
    assert torch.equal(inp["x_a"], _m0["x_a"]) and torch.equal(inp["tau_a"], _m0["tau_a"]), "the shift changed the audio"
    assert torch.equal(inp["x_s"], MATCHED_INPUTS[(o, "honest")]["x_s"]), "tau mode changed the sensor content"
del _m0
print(f"\n[INPUTS] {len(MATCHED_INPUTS)} matched-set input sets built ({len(OFFSETS_S)} offsets x {TAU_MODES}) | audio "
      f"identical in all ✅ | sensor content identical between tau modes ✅")
for o in OFFSETS_S:
    if o == 0.0:
        continue
    n_changed = int((MATCHED_INPUTS[(o, 'honest')]["x_s"] != MATCHED_INPUTS[(0.0, 'honest')]["x_s"]).flatten(1).any(1).sum())
    print(f"    offset {o:+.1f} s: x_s CONTENT changed in {n_changed} of {len(MATCHED_IDX)} windows | honest tau_s "
          f"[{float(MATCHED_INPUTS[(o, 'honest')]['tau_s'].min()):+.3f}, {float(MATCHED_INPUTS[(o, 'honest')]['tau_s'].max()):+.3f}]"
          f" | rezeroed tau_s [{float(MATCHED_INPUTS[(o, 'rezeroed')]['tau_s'].min()):+.3f}, "
          f"{float(MATCHED_INPUTS[(o, 'rezeroed')]['tau_s'].max()):+.3f}]")

# ------------------------------------------------------------------------------
# HOOK 4 — concrete tau example: what HTT/LACA see for ONE window at +1.0 s and -1.0 s
# ------------------------------------------------------------------------------
_p = len(MATCHED_IDX) // 2
print(f"\n[TAU EXAMPLE] matched window #{_p} ({TEST_INDEX[MATCHED_IDX[_p]][0]}, audio window start "
      f"{TEST_INDEX[MATCHED_IDX[_p]][1]:.3f} s), tau normalized by W = {WINDOW_SEC} s as the pipeline does:")
_ta = MATCHED_INPUTS[(0.0, "honest")]["tau_a"][_p]
for o in (+1.0, -1.0):
    ts_h, ts_z = MATCHED_INPUTS[(o, "honest")]["tau_s"][_p], MATCHED_INPUTS[(o, "rezeroed")]["tau_s"][_p]
    ts_0 = MATCHED_INPUTS[(0.0, "honest")]["tau_s"][_p]
    gap_h = float(ts_h.mean() - _ta.mean()) - float(ts_0.mean() - _ta.mean())
    gap_z = float(ts_z.mean() - _ta.mean()) - float(ts_0.mean() - _ta.mean())
    print(f"    offset {o:+.1f} s: tau_a [{float(_ta.min()):.3f}, {float(_ta.max()):.3f}] | honest tau_s [{float(ts_h.min()):+.3f}, "
          f"{float(ts_h.max()):+.3f}] | re-zeroed tau_s [{float(ts_z.min()):+.3f}, {float(ts_z.max()):+.3f}]")
    print(f"        mean(tau_s) - mean(tau_a), relative to offset 0: honest {gap_h:+.4f} (x W = {gap_h * WINDOW_SEC:+.3f} s, "
          f"expected {o / WINDOW_SEC:+.4f} = {o:+.1f} s) -> {'✅ reflects the injected offset' if abs(gap_h * WINDOW_SEC - o) < 0.05 else '🚨'}"
          f" | re-zeroed {gap_z:+.4f} (x W = {gap_z * WINDOW_SEC:+.3f} s) -> {'✅ carries no shift information' if abs(gap_z * WINDOW_SEC) < 0.05 else '🚨'}")
del _ta

## 8. Step 2b — shifted evaluation (no retraining)
Each model is loaded from its checkpoint, one at a time, and evaluated on:
- the **0 ms full test set**. This is also checked bit for bit against Step 1's `test_loader` probabilities.
- the **matched set** at every offset × {honest, re-zeroed} τ.

A row goes to `manipulation_check_results.csv` after every evaluation. It holds accuracy, macro-F1, balanced accuracy,
macro-AUROC, Class 3/4 recall and the window count. GPU memory is freed after each model.

**D τ-invariance:** D's honest and re-zeroed predictions must be **bit-identical** at ±1.0 s. It is checked at every
offset, and it validates the switches.

In [ ]:
# ==============================================================================
# STEP 2b — SHIFTED EVALUATION, model by model (no retraining). Every (condition, seed, offset, tau mode) on the
# matched set + the 0 ms full test set; a CSV row after EVERY evaluation; GPU freed after each model.
# ==============================================================================
RESULTS_FIELDS = ["eval_no", "condition", "seed", "offset_s", "tau_mode", "eval_set", "model_source", "n_windows"] + METRICS + \
                 ["class3_recall", "class4_recall", "identical_to_honest", "confusion_matrix", "finished_at"]


def eval_inputs(model, inputs):
    """v25: softmax probabilities in input order — batches of BATCH_SIZE in test-loader order, eval mode, no RNG."""
    model.eval()
    n, probs = len(inputs["label"]), []
    with torch.no_grad():
        for i in range(0, n, BATCH_SIZE):
            xa, xs, ta, ts = (inputs[k][i:i + BATCH_SIZE].to(device) for k in ("x_a", "x_s", "tau_a", "tau_s"))
            probs.append(F.softmax(model(xa, xs, ta, ts)[0], dim=-1).cpu().numpy())
    return np.concatenate(probs)


RESULTS, PROBS = [], {}      # PROBS[(cond, seed, offset, tau_mode)] = matched-set probabilities
EVAL_PATH_MATCH = {}         # (cond, seed) -> 0 ms full-test probs here == Step 1's test_loader probs (bit-identical)
D_TAU_INVARIANT = {}         # (seed, offset) -> D honest vs re-zeroed predictions bit-identical
N_EVALS_TOTAL = len(RUN_ORDER) * (1 + len(OFFSETS_S) * len(TAU_MODES))


def log_eval(cond, seed, offset, tau_mode, eval_set, labels, probs, identical_to_honest=None):
    ev = evaluate_probs(labels, probs)
    row = {"eval_no": len(RESULTS) + 1, "condition": cond, "seed": seed, "offset_s": offset, "tau_mode": tau_mode,
           "eval_set": eval_set, "model_source": MODEL_SOURCES[(cond, seed)], "n_windows": len(labels),
           **{m: ev[m] for m in METRICS}, "class3_recall": ev["class3_recall"], "class4_recall": ev["class4_recall"],
           "identical_to_honest": identical_to_honest,
           "confusion_matrix": json.dumps(np.asarray(ev["confusion_matrix"]).tolist()),
           "finished_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
    RESULTS.append(row)
    _append(RESULTS_CSV, RESULTS_FIELDS, row)
    return row


t_step2 = time.time()
for slot, (seed, cond) in enumerate(RUN_ORDER, start=1):
    htt_on, laca_on = CONDITIONS[cond]
    tag = f"[{cond} seed={seed}]"
    model = AeroSyncMamba(htt_time=htt_on, laca_lag=laca_on).to(device)
    model.load_state_dict(torch.load(BASELINES[(cond, seed)]["checkpoint"], map_location=device))
    verify_architecture(model)
    model.eval()

    full_probs = eval_inputs(model, FULL0)
    EVAL_PATH_MATCH[(cond, seed)] = bool(np.array_equal(full_probs, BASELINES[(cond, seed)]["test_probs"]))
    r = log_eval(cond, seed, 0.0, "honest", "full", TEST_LABELS, full_probs)
    print(f"{tag} 0 ms FULL test set ({N_TEST}): macro-F1 {r['macro_f1']:.4f} | eval path == Step 1 "
          + ("bit-identical ✅" if EVAL_PATH_MATCH[(cond, seed)] else "⚠️ differs from Step 1's test_loader probabilities"))

    for o in OFFSETS_S:
        for mode in TAU_MODES:
            probs = eval_inputs(model, MATCHED_INPUTS[(o, mode)])
            PROBS[(cond, seed, o, mode)] = probs
            same = None if mode == "honest" else bool(np.array_equal(probs, PROBS[(cond, seed, o, "honest")]))
            log_eval(cond, seed, o, mode, "matched", MATCHED_LABELS, probs, same)
        if cond == "D" and o != 0.0:
            D_TAU_INVARIANT[(seed, o)] = bool(np.array_equal(PROBS[("D", seed, o, "honest")], PROBS[("D", seed, o, "rezeroed")]))
    rows = {(x["offset_s"], x["tau_mode"]): x for x in RESULTS
            if x["condition"] == cond and x["seed"] == seed and x["eval_set"] == "matched"}
    print(f"{tag} MATCHED set ({len(MATCHED_IDX)}) macro-F1 / balanced acc: " + " | ".join(
        f"{o:+.1f}s " + "/".join(f"{mode[0]}:{rows[(o, mode)]['macro_f1']:.4f},{rows[(o, mode)]['balanced_accuracy']:.4f}"
                                 for mode in TAU_MODES) for o in OFFSETS_S))
    if cond == "D":
        _inv = {o: D_TAU_INVARIANT[(seed, o)] for o in OFFSETS_S if o != 0.0}
        print(f"{tag} [D TAU INVARIANCE] honest vs re-zeroed tau -> predictions " + ", ".join(
            f"{o:+.1f}s {'bit-identical ✅' if v else 'CHANGED 🚨'}" for o, v in _inv.items()))
        if not all(_inv.values()):
            print(f"🚨🚨🚨 {tag} D reads no timestamps, so its predictions must not depend on tau. This is a BUG in the "
                  f"switches or the injection, not a finding. 🚨🚨🚨")
    else:
        print(f"{tag} [C] re-zeroed tau changes C's predictions at: " + ", ".join(
            f"{o:+.1f}s {'no' if rows[(o, 'rezeroed')]['identical_to_honest'] else 'yes'}" for o in OFFSETS_S if o != 0.0)
              + " (expected yes: C reads tau)")

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f">>> {slot}/{len(RUN_ORDER)} models evaluated | {len(RESULTS)}/{N_EVALS_TOTAL} evaluations logged to "
          f"{RESULTS_CSV.name} | elapsed {(time.time() - t_step2) / 60:.1f} min")

D_TAU_INVARIANCE_PASS = all(D_TAU_INVARIANT[(s, o)] for s in SEEDS for o in DECISION_OFFSETS_S)
print("\n" + "█" * 110)
print(f"█  D TAU-INVARIANCE at {DECISION_OFFSETS_S} s (honest vs re-zeroed, all {len(SEEDS)} seeds): "
      + ("✅ PASS — predictions bit-identical; the switches work" if D_TAU_INVARIANCE_PASS else
         "🚨 FAIL — a timestamp path is active in Condition D"))
print("█" * 110)

## 9. Step 3 — drops, paired differences and the pre-registered verdict
- **Drop** = the model's own 0 ms matched-set score minus its score at the offset (positive = worse), for macro-F1 and
  balanced accuracy. Each direction is reported separately, as mean ± SD (ddof = 1) over the 3 seeds.
- **Paired** (D drop − C drop) per seed, with mean, SD and wins/3, for C with honest τ and for C with re-zeroed τ.
- **Fixed thresholds** (not tuned after seeing results):
  - D *degrades* if the mean macro-F1 drop ≥ 0.015, the mean > the SD of the drops, and the drop is > 0 on all 3 seeds.
  - C *retains more* if the mean (D − C) ≥ 0.015, the mean > the SD of the paired differences, and it is > 0 on all 3
    seeds. This uses honest τ; re-zeroed τ is a diagnostic.
- **Decision at ±1.0 s**, each direction separately; ±0.5 s is informational. The overall verdict takes the most
  favourable direction:
  - **1:** D does not degrade;
  - **2:** D degrades, but C does not retain more;
  - **3:** both hold.

With n = 3, every SD is crude, so the verdict is a screening decision.

In [ ]:
# ==============================================================================
# STEP 3 — DROPS from each model's own 0 ms score on the MATCHED set, paired (D drop - C drop), pre-registered verdict
# ==============================================================================
DROP_METRICS = [("macro-F1", "macro_f1"), ("balanced accuracy", "balanced_accuracy")]
SHIFTED = [o for o in OFFSETS_S if o != 0.0]
DROPS_CSV = WORK_DIR / "manipulation_check_drops.csv"
_score = {(x["condition"], x["seed"], x["offset_s"], x["tau_mode"]): x for x in RESULTS if x["eval_set"] == "matched"}


def drops(cond, mode, o, k):
    """Per-seed drop (positive = worse) from the model's own 0 ms matched-set score; the 0 ms base is tau-mode free."""
    return np.array([_score[(cond, s, 0.0, "honest")][k] - _score[(cond, s, o, mode)][k] for s in SEEDS])


def summ(v):
    return {"vals": v, "mean": float(v.mean()), "sd": float(v.std(ddof=1)), "pos": int((v > 0).sum())}


def passes(s, threshold):
    """Pre-registered rule: mean >= threshold AND mean > across-seed SD AND positive on every seed."""
    return s["mean"] >= threshold and s["mean"] > s["sd"] and s["pos"] == len(SEEDS)


ANALYSIS, _drop_rows = {}, []
for label, k in DROP_METRICS:
    print("\n" + "=" * 128)
    print(f"DROPS in {label.upper()} from each model's own 0 ms score, matched set ({len(MATCHED_IDX)} windows); positive = "
          f"worse | mean ± SD over {len(SEEDS)} seeds (SD ddof=1) | per seed {SEEDS}")
    print("=" * 128)
    print(f"    {'offset':>7}  {'D drop':<34}{'C drop (honest tau)':<34}{'C drop (re-zeroed tau)':<34}")
    for o in SHIFTED:
        d, ch, cz = summ(drops("D", "honest", o, k)), summ(drops("C", "honest", o, k)), summ(drops("C", "rezeroed", o, k))
        pair_h, pair_z = summ(d["vals"] - ch["vals"]), summ(d["vals"] - cz["vals"])
        ANALYSIS[(k, o)] = {"D": d, "C_honest": ch, "C_rezeroed": cz, "paired_honest": pair_h, "paired_rezeroed": pair_z}
        fmt = lambda s: f"{s['mean']:+.4f} ± {s['sd']:.4f} [" + ",".join(f"{x:+.3f}" for x in s["vals"]) + "]"
        print(f"    {o:>+6.1f}s  {fmt(d):<34}{fmt(ch):<34}{fmt(cz):<34}")
        for s_i, s in enumerate(SEEDS):
            _drop_rows.append({"metric": k, "offset_s": o, "seed": s, "D_drop": d["vals"][s_i], "C_honest_drop": ch["vals"][s_i],
                               "C_rezeroed_drop": cz["vals"][s_i], "paired_honest": pair_h["vals"][s_i],
                               "paired_rezeroed": pair_z["vals"][s_i]})
    print(f"\n    PAIRED (D drop - C drop) per seed — positive = C lost LESS than D")
    print(f"    {'offset':>7}  {'C honest: mean ± SD [per seed]':<44}{'wins':>6}   {'C re-zeroed: mean ± SD [per seed]':<44}{'wins':>6}")
    for o in SHIFTED:
        a = ANALYSIS[(k, o)]
        for_p = lambda s: f"{s['mean']:+.4f} ± {s['sd']:.4f} [" + ",".join(f"{x:+.3f}" for x in s["vals"]) + "]"
        print(f"    {o:>+6.1f}s  {for_p(a['paired_honest']):<44}{a['paired_honest']['pos']:>4}/{len(SEEDS)}   "
              f"{for_p(a['paired_rezeroed']):<44}{a['paired_rezeroed']['pos']:>4}/{len(SEEDS)}")
with open(DROPS_CSV, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(_drop_rows[0]))
    w.writeheader()
    w.writerows(_drop_rows)
print(f"\n[per-seed drops written to {DROPS_CSV}]")

# --- context: Class 3 / Class 4 recall drops (not part of the decision rule) ---
print("\n[CONTEXT] Class 3 / Class 4 recall drop (mean over seeds, honest tau): " + " | ".join(
    f"{o:+.1f}s D {drops('D', 'honest', o, 'class3_recall').mean():+.3f}/{drops('D', 'honest', o, 'class4_recall').mean():+.3f}, "
    f"C {drops('C', 'honest', o, 'class3_recall').mean():+.3f}/{drops('C', 'honest', o, 'class4_recall').mean():+.3f}"
    for o in SHIFTED))

# ------------------------------------------------------------------------------
# PRE-REGISTERED DECISION (macro-F1; decided at +/-1.0 s, each direction separately; +/-0.5 s informational)
# ------------------------------------------------------------------------------
def rule_outcome(o, c_mode="honest"):
    a = ANALYSIS[("macro_f1", o)]
    d_deg = passes(a["D"], DEGRADE_MIN_DROP)
    c_ret = passes(a[f"paired_{c_mode}"], RETAIN_MIN_DIFF)
    return d_deg, c_ret, (1 if not d_deg else (3 if c_ret else 2))


print("\n" + "█" * 128)
print("█  PRE-REGISTERED RULE (fixed before results): D DEGRADES at an offset if mean macro-F1 drop >= "
      f"{DEGRADE_MIN_DROP} AND > SD of drops AND > 0 on all {len(SEEDS)} seeds;")
print(f"█  C RETAINS MORE if mean (D drop - C drop) >= {RETAIN_MIN_DIFF} AND > SD of the paired differences AND > 0 on all "
      f"{len(SEEDS)} seeds. Primary = C with HONEST tau; re-zeroed tau is a diagnostic only.")
print("█" * 128)
for o in SHIFTED:
    a = ANALYSIS[("macro_f1", o)]
    d_deg, c_ret, br = rule_outcome(o)
    _, c_ret_z, _ = rule_outcome(o, "rezeroed")
    role = "DECISION" if o in DECISION_OFFSETS_S else "informational"
    direction = "IMU delayed" if o > 0 else "audio delayed"
    print(f"█  {o:+.1f} s ({direction:<13}, {role:<13}): D drop {a['D']['mean']:+.4f} ± {a['D']['sd']:.4f} "
          f"({a['D']['pos']}/{len(SEEDS)} > 0) -> D {'DEGRADES' if d_deg else 'does not degrade'} | D-C (honest) "
          f"{a['paired_honest']['mean']:+.4f} ± {a['paired_honest']['sd']:.4f} ({a['paired_honest']['pos']}/{len(SEEDS)}) -> C "
          f"{'RETAINS MORE' if c_ret else 'does not retain more'} | re-zeroed diag: {'retains more' if c_ret_z else 'no'} "
          f"-> branch {br}")
DIRECTION_BRANCH = {o: rule_outcome(o)[2] for o in DECISION_OFFSETS_S}
BRANCH = max(DIRECTION_BRANCH.values())
_best = [o for o in DECISION_OFFSETS_S if DIRECTION_BRANCH[o] == BRANCH]
DECIDING_OFFSET = max(_best, key=lambda o: ANALYSIS[("macro_f1", o)]["D"]["mean"])
print("█" * 128)
print(f"█  Per direction at +/-1.0 s: " + ", ".join(f"{o:+.1f} s -> branch {b}" for o, b in DIRECTION_BRANCH.items())
      + f" | overall = the most favourable direction: {DECIDING_OFFSET:+.1f} s -> BRANCH {BRANCH}")
print("█" * 128)
if BRANCH == 1:
    print("█  BRANCH 1: MISALIGNMENT UP TO 1.0 s DOES NOT MEASURABLY HURT D ON THIS DATASET. An alignment mechanism has")
    print("█  nothing to recover here; recommend stopping the LACA/temporal-alignment direction for TII.")
elif BRANCH == 2:
    _worst = max((o for o in DECISION_OFFSETS_S if DIRECTION_BRANCH[o] >= 2), key=lambda o: ANALYSIS[("macro_f1", o)]["D"]["mean"])
    print("█  BRANCH 2: D DEGRADES, BUT C DOES NOT RETAIN MORE. An eval-only test cannot credit LACA: C's lag bias never saw")
    print("█  any lag in training (it was trained on aligned data only), so it had nothing to learn to compensate for.")
    print(f"█  RECOMMENDATION: one targeted TRAIN-TIME run at the worst offset only ({_worst:+.1f} s, D mean macro-F1 drop "
          f"{ANALYSIS[('macro_f1', _worst)]['D']['mean']:+.4f}): {len(SEEDS)} seeds x 2 conditions. NOT implemented or run here.")
else:
    print("█  BRANCH 3: REAL ROBUSTNESS SIGNAL, the full controlled misalignment study is justified.")
print("█" * 128)
print(f"█  CAVEAT: n = {len(SEEDS)} seeds makes every SD crude (2 degrees of freedom). This verdict is a SCREENING decision,")
print("█  not a final statistical claim. Seed-to-seed SD of macro-F1 in this project is ~0.016 for context.")
print("█" * 128)

## 10. Final validation
A recap of every check:
- the data is reused exactly (CORRECTED == v23);
- the D reproduction;
- the re-slice at offset 0;
- the achieved offsets;
- the evaluation path;
- D τ-invariance;
- the evaluation count, window retention and the verdict.

In [ ]:
# ==============================================================================
# FINAL VALIDATION — every verification hook in one place
# ==============================================================================
_trained = {k: r for k, r in BASELINES.items() if MODEL_SOURCES[k] == "trained in v26"}
print("=" * 110)
print("FINAL VALIDATION")
print("=" * 110)
checks = {
    "CORRECTED code == v23 (source sha1)": False not in CORRECTED_SOURCE_CHECK.values(),
    "CORRECTED missions + windows + class counts == v23": all(CORRECTED_DATA_CHECK.values()),
    "every trained run starts at its seed's reference weights + RNG": all(r["paired_init"] for r in _trained.values()),
    "Step 1: D reproduces v24 macro-F1 (±0.0005), every seed": D_REPRO_ALL_PASS,
    "Step 2: offset-0 re-slice == stored test windows": RESLICE_PASS,
    "Step 2: achieved offsets within 50 ms of requested": ACHIEVED_OK,
    "Step 2: evaluation path == Step 1 test_loader (all models)": all(EVAL_PATH_MATCH.values()),
    "Step 2: D tau-invariance (honest == re-zeroed) at +/-1.0 s": D_TAU_INVARIANCE_PASS,
    f"all {N_EVALS_TOTAL} evaluations logged": len(RESULTS) == N_EVALS_TOTAL,
    "fusion backbone is real mamba_ssm": MAMBA_CLASS is not None,
}
for name, ok in checks.items():
    print(f"  {name:<64}: {'✅ pass' if ok else '🚨 FAIL'}")
print(f"  {'D models':<64}: " + ", ".join(f"seed {s} {MODEL_SOURCES[('D', s)]}" for s in SEEDS))
print(f"  {'C models (new CORRECTED baseline)':<64}: " + ", ".join(
    f"seed {s} {BASELINES[('C', s)]['macro_f1']:.4f}{'' if C_SANITY[s][1] else ' ⚠️'}" for s in SEEDS))
print(f"  {'test windows kept at +/-0.5 / +/-1.0 s':<64}: " + ", ".join(
    f"{o:+.1f}s {RETENTION[o]['kept']}/{N_TEST} (-{RETENTION[o]['lost_pct']:.2f}%)" for o in SHIFTED))
print(f"  {'matched set (valid at every offset)':<64}: {len(MATCHED_IDX)}/{N_TEST}")
print(f"  {'VERDICT':<64}: BRANCH {BRANCH} (decided at {DECIDING_OFFSET:+.1f} s; n = {len(SEEDS)} seeds, screening only)")
print(f"  {'outputs':<64}: {RESULTS_CSV.name}, {TRAINING_CSV.name}, {DROPS_CSV.name}, {CHECKPOINT_DIR.name}/")
print("=" * 110)
if not all(checks.values()):
    print("🚨 at least one check FAILED — read the flagged line before using the verdict.")